<a href="https://colab.research.google.com/github/psar0006/NBA_Models/blob/main/NBA_Pts_Model_V1_Cleaned.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NBA Player Points Prediction — V1
**Research notebook | LightGBM conditional mean + Negative Binomial probabilities**

This notebook preserves the original modelling experiments and their results, while organising the workflow from historical feature engineering through model selection, probabilistic calibration, diagnostics, and V1 model export.

**Reading guide:** Run cells in order within each stage. Some experiments are alternatives rather than steps required for production inference. Existing code and outputs have been retained; the added explanations do not imply that the models have been rerun or independently validated.

**V1 scope:** The points model and conditional dispersion model are retained as developed. Integrating separately predicted pregame minutes is deferred to V2.

---
## 1. Data ingestion and pregame feature engineering
Prepare historical player-game data and construct predictors based on previous games.

In [ ]:
##Import Data

import polars as pl
import numpy as np
import kagglehub
import datetime as dt
from kagglehub import KaggleDatasetAdapter



lf_player_stats = kagglehub.dataset_load(
    KaggleDatasetAdapter.POLARS,
    "eoinamoore/historical-nba-data-and-player-box-scores",
    "PlayerStatisticsExtended.csv",
    polars_kwargs={
        "schema_overrides": {
            "seriesGameNumber": pl.String,
            "numMinutes": pl.String,

            # Basic box score statistics
            "points": pl.Float64,
            "reboundsDefensive": pl.Float64,
            "reboundsOffensive": pl.Float64,
            "reboundsTotal": pl.Float64,
            "foulsPersonal": pl.Float64,
            "assists": pl.Float64,
            "steals": pl.Float64,
            "blocks": pl.Float64,
            "turnovers": pl.Float64,

            # Shooting
            "fieldGoals": pl.Float64,
            "fieldGoalsAttempted": pl.Float64,
            "fieldGoalsMade": pl.Float64,
            "fieldGoalsPercentage": pl.Float64,

            "threePointersMade": pl.Float64,
            "threePointersAttempted": pl.Float64,
            "threePointersPercentage": pl.Float64,

            "freeThrowsMade": pl.Float64,
            "freeThrowsAttempted": pl.Float64,
            "freeThrowsPercentage": pl.Float64,

            "plusMinusPoints": pl.Float64,

            # Advanced statistics
            "offensiveRating": pl.Float64,
            "defensiveRating": pl.Float64,
            "assistPercentage": pl.Float64,
            "assistToTurnoverRatio": pl.Float64,
            "assistRatio": pl.Float64,
            "offensiveReboundPercentage": pl.Float64,
            "defensiveReboundPercentage": pl.Float64,
            "reboundPercentage": pl.Float64,
            "estimatedTurnoverPercentage": pl.Float64,
            "effectiveFieldGoalPercentage": pl.Float64,
            "usagePercentage": pl.Float64,
            "pace": pl.Float64,
            "pacePer40": pl.Float64,

            # Percentage of team production
            "percentTeamOffensiveRebounds": pl.Float64,
            "percentTeamDefensiveRebounds": pl.Float64,
            "percentTeamBlocks": pl.Float64,
            "percentTeamPoints": pl.Float64,
        }
    }
)





lf_team_stats = kagglehub.dataset_load(
  KaggleDatasetAdapter.POLARS,
  "eoinamoore/historical-nba-data-and-player-box-scores",
  "TeamStatisticsExtended.csv"

  # Provide any additional arguments like
  # sql_query, polars_frame_type, or
  # polars_kwargs.
  # See the documenation for more information:
  # https://github.com/Kaggle/kagglehub/blob/main/README.md#kaggledatasetadapterpolars
)


100%|██████████| 432M/432M [00:04<00:00, 92.9MB/s]


100%|██████████| 36.4M/36.4M [00:00<00:00, 78.4MB/s]


### 1.1 Initial data cleaning
Standardise source fields and establish the base player-game dataset used downstream.

In [ ]:
## ============================================================
# BASIC CLEANING OF DATA
# ============================================================


# ============================================================
# 1. CONVERT numMinutes FROM MM:SS TO DECIMAL MINUTES
# ============================================================

lf_player_stats = lf_player_stats.with_columns(
    pl.when(
        pl.col("numMinutes").is_not_null()
        & pl.col("numMinutes").str.contains(":")
    )
    .then(
        pl.col("numMinutes")
        .str.split_exact(":", 1)
        .struct.field("field_0")
        .cast(pl.Float64)
        +
        (
            pl.col("numMinutes")
            .str.split_exact(":", 1)
            .struct.field("field_1")
            .cast(pl.Float64)
            / 60
        )
    )
    .otherwise(
        pl.col("numMinutes").cast(pl.Float64, strict=False)
    )
    .alias("numMinutes")
)


# ============================================================
# 2. SELECT RELEVANT COLUMNS
# ============================================================

lf_player_stats = lf_player_stats.select(

    # --------------------------------------------------------
    # Identifiers
    # --------------------------------------------------------
    "personId",
    "gameId",
    "gameDateTimeEst",

    # --------------------------------------------------------
    # Team / opponent
    # --------------------------------------------------------
    "playerteamCity",
    "playerteamName",
    "playerteamId",

    "opponentteamCity",
    "opponentteamName",
    "opponentteamId",

    "home",

    # --------------------------------------------------------
    # Basic player box score
    # --------------------------------------------------------
    "numMinutes",
    "points",
    "assists",
    "blocks",
    "steals",

    "fieldGoalsAttempted",
    "fieldGoalsMade",
    "fieldGoalsPercentage",

    "threePointersAttempted",
    "threePointersMade",
    "threePointersPercentage",

    "freeThrowsAttempted",
    "freeThrowsMade",
    "freeThrowsPercentage",

    "reboundsDefensive",
    "reboundsOffensive",
    "reboundsTotal",

    "turnovers",
    "foulsPersonal",
    "plusMinusPoints",

    # --------------------------------------------------------
    # Advanced player statistics
    # --------------------------------------------------------
    "offensiveRating",
    "defensiveRating",

    "assistPercentage",
    "assistToTurnoverRatio",
    "assistRatio",

    "offensiveReboundPercentage",
    "defensiveReboundPercentage",
    "reboundPercentage",

    "estimatedTurnoverPercentage",
    "effectiveFieldGoalPercentage",

    "usagePercentage",

    "pace",
    "pacePer40",

    # --------------------------------------------------------
    # Percentage of team production
    # --------------------------------------------------------
    "percentTeamOffensiveRebounds",
    "percentTeamDefensiveRebounds",
    "percentTeamBlocks",
    "percentTeamPoints",

    # --------------------------------------------------------
    # Player / game context
    # --------------------------------------------------------
    "startingPosition",
    "gameType",
    "comment"
)


# ============================================================
# 3. CONVERT GAME DATE
# ============================================================

lf_player_stats = (
    lf_player_stats
    .with_columns(
        pl.col("gameDateTimeEst")
        .str.to_datetime("%Y-%m-%d %H:%M:%S")
        .dt.date()
    )
    .filter(
        pl.col("gameDateTimeEst") >= dt.date(1996, 11, 1)
    )
)


# ============================================================
# 4. CREATE STARTER INDICATOR
# ============================================================

lf_player_stats = lf_player_stats.with_columns(
    (
        pl.col("startingPosition").is_not_null()
        &
        (pl.col("startingPosition") != "")
    )
    .cast(pl.Int8)
    .alias("is_starter")
)


# ============================================================
# 5. CREATE PRE-GAME AVAILABILITY INDICATOR
# ============================================================
# Temporary historical proxy until NBA lineup rosterStatus
# data is integrated.
#
# 1 = Available
# 0 = Unavailable due to injury / illness etc.
# ============================================================

lf_player_stats = lf_player_stats.with_columns(

    pl.when(
        pl.col("comment")
        .fill_null("")
        .str.to_lowercase()
        .str.contains(
            r"injur|illness|sick|concussion|sprain|strain|soreness|"
            r"fracture|broken|surgery|rehab|recovery|contusion|"
            r"tendinitis|tendonitis|inflammation|infection|"
            r"back spasms|personal reasons"
        )
    )
    .then(0)
    .otherwise(1)
    .cast(pl.Int8)
    .alias("pregame_available")
)


# ============================================================
# 6. KEEP REGULAR-SEASON GAMES ONLY
# ============================================================

lf_player_stats = lf_player_stats.filter(
    ~pl.col("gameType").is_in([
        "Pre Season",
        "Post Season",
        "All-Star Game",
        "Preseason",
        "Playoffs"
    ])
)

### 1.2 Historical offensive and form features
Construct lagged, rolling and exponentially weighted player statistics. Confirm that each predictor is available **before** the game being forecast.

In [ ]:
# ============================================================
# PLAYER POINTS MODEL - PRE-GAME FEATURE ENGINEERING
# ============================================================

# IMPORTANT:
# All historical features use shift(1), ensuring the current
# game's information is never included in its own predictors.


# ============================================================
# 1. SORT DATA CHRONOLOGICALLY
# ============================================================

lf_player_stats = (
    lf_player_stats
    .sort([
        "personId",
        "gameDateTimeEst",
        "gameId"
    ])
)


# ============================================================
# 2. DEFINE METRICS
# ============================================================

# Minutes / playing opportunity
MINUTES_METRICS = [
    "numMinutes",
]

# Core scoring / shot-volume statistics
SCORING_METRICS = [
    "points",
    "fieldGoalsAttempted",
    "fieldGoalsMade",
    "freeThrowsAttempted",
    "freeThrowsMade",
    "threePointersAttempted",
    "threePointersMade",
]

# Offensive context
CONTEXT_METRICS = [
    "pace",
    "usagePercentage",
]

# All metrics used for historical form features
FORM_METRICS = (
    MINUTES_METRICS
    + SCORING_METRICS
    + CONTEXT_METRICS
)


# ============================================================
# 3. PREVIOUS GAME VALUES
# ============================================================

lf_player_stats = lf_player_stats.with_columns([

    pl.col(metric)
    .shift(1)
    .over("personId")
    .alias(f"{metric}_lag1")

    for metric in FORM_METRICS

])


# ============================================================
# 4. EWMA FEATURES
# ============================================================
# alpha = 0.60
# More weight is given to recent performances.

lf_player_stats = lf_player_stats.with_columns([

    pl.col(metric)
    .shift(1)
    .ewm_mean(
        alpha=0.60,
        adjust=False,
        ignore_nulls=True
    )
    .over("personId")
    .alias(f"{metric}_ewma")

    for metric in FORM_METRICS

])


# ============================================================
# 5. ROLLING 5-GAME AVERAGES
# ============================================================

lf_player_stats = lf_player_stats.with_columns([

    pl.col(metric)
    .shift(1)
    .rolling_mean(
        window_size=5,
        min_samples=1
    )
    .over("personId")
    .alias(f"{metric}_avg5")

    for metric in FORM_METRICS

])


# ============================================================
# 6. ROLLING 10-GAME AVERAGES
# ============================================================

lf_player_stats = lf_player_stats.with_columns([

    pl.col(metric)
    .shift(1)
    .rolling_mean(
        window_size=10,
        min_samples=1
    )
    .over("personId")
    .alias(f"{metric}_avg10")

    for metric in FORM_METRICS

])


# ============================================================
# 7. TREND FEATURES
# ============================================================

lf_player_stats = lf_player_stats.with_columns([

    # Points trend relative to EWMA
    (
        pl.col("points_avg5")
        - pl.col("points_ewma")
    )
    .alias("points_trend_5v_ewma"),

    # Minutes / rotation trend
    (
        pl.col("numMinutes_avg5")
        - pl.col("numMinutes_avg10")
    )
    .alias("minutes_trend_5v10"),

    # Usage / offensive role trend
    (
        pl.col("usagePercentage_avg5")
        - pl.col("usagePercentage_avg10")
    )
    .alias("usage_trend_5v10"),

    # Shot-volume trend
    (
        pl.col("fieldGoalsAttempted_avg5")
        - pl.col("fieldGoalsAttempted_avg10")
    )
    .alias("fga_trend_5v10"),

    # Three-point volume trend
    (
        pl.col("threePointersAttempted_avg5")
        - pl.col("threePointersAttempted_avg10")
    )
    .alias("threepa_trend_5v10"),

    # Free-throw volume trend
    (
        pl.col("freeThrowsAttempted_avg5")
        - pl.col("freeThrowsAttempted_avg10")
    )
    .alias("fta_trend_5v10"),

])

### 1.3 Opponent and defensive context
Add season-aware defensive profiles and prior-game context, including opponent-related features. Any same-game roster or availability inputs require separate pregame availability checks before deployment.

In [ ]:
# ============================================================
# DEFENSIVE PROFILE FEATURE ENGINEERING
# ============================================================


# ============================================================
# 1. CREATE NBA SEASON
# ============================================================

lf_player_stats = lf_player_stats.with_columns(

    pl.when(
        pl.col("gameDateTimeEst").dt.month() >= 10
    )
    .then(
        pl.col("gameDateTimeEst").dt.year()
    )
    .otherwise(
        pl.col("gameDateTimeEst").dt.year() - 1
    )
    .alias("season")

)


# ============================================================
# 2. PLAYER DEFENSIVE RATING HISTORY
# ============================================================
# IMPORTANT:
# All defensive rating features are shifted by one game.
# Therefore the current game's defensiveRating is never used
# to predict the current game.
#
# Features reset each NBA season.
# ============================================================

lf_player_stats = (
    lf_player_stats
    .sort([
        "personId",
        "season",
        "gameDateTimeEst",
        "gameId",
    ])
)


lf_player_stats = lf_player_stats.with_columns([

    # --------------------------------------------------------
    # Previous game
    # --------------------------------------------------------

    pl.col("defensiveRating")
    .shift(1)
    .over([
        "personId",
        "season",
    ])
    .alias("defensiveRating_lag1"),


    # --------------------------------------------------------
    # EWMA
    # --------------------------------------------------------

    pl.col("defensiveRating")
    .shift(1)
    .ewm_mean(
        alpha=0.60,
        adjust=False,
        ignore_nulls=True,
    )
    .over([
        "personId",
        "season",
    ])
    .alias("defensiveRating_ewma"),


    # --------------------------------------------------------
    # Last 5
    # --------------------------------------------------------

    pl.col("defensiveRating")
    .shift(1)
    .rolling_mean(
        window_size=5,
        min_samples=1,
    )
    .over([
        "personId",
        "season",
    ])
    .alias("defensiveRating_avg5"),


    # --------------------------------------------------------
    # Last 10
    # --------------------------------------------------------

    pl.col("defensiveRating")
    .shift(1)
    .rolling_mean(
        window_size=10,
        min_samples=1,
    )
    .over([
        "personId",
        "season",
    ])
    .alias("defensiveRating_avg10"),

])


# ============================================================
# 3. CREATE TEAM-GAME POINTS SCORED
# ============================================================
# Collapse player box scores into one row per team per game.
# ============================================================

lf_team_game = (
    lf_player_stats

    .group_by([
        "season",
        "gameId",
        "gameDateTimeEst",
        "playerteamId",
        "opponentteamId",
    ])

    .agg(
        pl.col("points")
        .sum()
        .alias("team_points")
    )
)


# ============================================================
# 4. CREATE TEAM POINTS ALLOWED
# ============================================================
# Take the opponent's team_points and attach it to the
# corresponding team-game row.
#
# Example:
#
# BOS 110 vs NYK 105
#
# BOS points_allowed = 105
# NYK points_allowed = 110
# ============================================================

lf_opponent_score = (
    lf_team_game

    .select([
        "season",
        "gameId",

        pl.col("playerteamId")
        .alias("opponentteamId"),

        pl.col("team_points")
        .alias("points_allowed"),
    ])
)


lf_team_game = (
    lf_team_game

    .join(
        lf_opponent_score,

        on=[
            "season",
            "gameId",
            "opponentteamId",
        ],

        how="left",
    )
)


# ============================================================
# 5. HISTORICAL TEAM POINTS ALLOWED
# ============================================================
# All features use PRIOR games only.
#
# Reset each season.
# ============================================================

lf_team_game = (
    lf_team_game

    .sort([
        "playerteamId",
        "season",
        "gameDateTimeEst",
        "gameId",
    ])
)


lf_team_game = lf_team_game.with_columns([

    # Previous game
    pl.col("points_allowed")
    .shift(1)
    .over([
        "playerteamId",
        "season",
    ])
    .alias("points_allowed_lag1"),


    # EWMA
    pl.col("points_allowed")
    .shift(1)
    .ewm_mean(
        alpha=0.60,
        adjust=False,
        ignore_nulls=True,
    )
    .over([
        "playerteamId",
        "season",
    ])
    .alias("points_allowed_ewma"),


    # Last 5
    pl.col("points_allowed")
    .shift(1)
    .rolling_mean(
        window_size=5,
        min_samples=1,
    )
    .over([
        "playerteamId",
        "season",
    ])
    .alias("points_allowed_avg5"),


    # Last 10
    pl.col("points_allowed")
    .shift(1)
    .rolling_mean(
        window_size=10,
        min_samples=1,
    )
    .over([
        "playerteamId",
        "season",
    ])
    .alias("points_allowed_avg10"),


    # Last 20
    pl.col("points_allowed")
    .shift(1)
    .rolling_mean(
        window_size=20,
        min_samples=1,
    )
    .over([
        "playerteamId",
        "season",
    ])
    .alias("points_allowed_avg20"),

])


# ------------------------------------------------------------
# Recent defensive trend
#
# Positive:
# team is allowing MORE points recently
#
# Negative:
# team is allowing FEWER points recently
# ------------------------------------------------------------

lf_team_game = lf_team_game.with_columns(

    (
        pl.col("points_allowed_avg5")
        - pl.col("points_allowed_avg20")
    )
    .alias("points_allowed_trend_5v20")

)


# ============================================================
# 6. PLAYER EXPECTED DEFENSIVE CONTRIBUTION
# ============================================================
#
# Historical player defensive quality × expected minutes.
#
# numMinutes_ewma must itself be calculated using previous
# games only.
# ============================================================

lf_player_stats = lf_player_stats.with_columns(

    (
        pl.col("defensiveRating_ewma")
        * pl.col("numMinutes_ewma")
    )
    .alias("defensive_rating_weighted_component")

)


# ============================================================
# 7. EXPECTED FULL ROTATION DEFENSIVE RATING
# ============================================================
#
# Represents expected defensive quality BEFORE accounting
# for today's unavailable players.
#
# Weighted by each player's expected minutes.
# ============================================================

lf_rotation_defence = (
    lf_player_stats

    .filter(
        pl.col("numMinutes_ewma").is_not_null()
        &
        pl.col("defensiveRating_ewma").is_not_null()
        &
        (pl.col("numMinutes_ewma") > 0)
    )

    .group_by([
        "season",
        "gameId",
        "playerteamId",
    ])

    .agg([

        pl.col("defensive_rating_weighted_component")
        .sum()
        .alias("weighted_drtg_sum"),


        pl.col("numMinutes_ewma")
        .sum()
        .alias("expected_rotation_minutes"),


        pl.col("personId")
        .n_unique()
        .alias("rotation_player_count"),

    ])

    .with_columns(

        pl.when(
            pl.col("expected_rotation_minutes") > 0
        )
        .then(
            pl.col("weighted_drtg_sum")
            / pl.col("expected_rotation_minutes")
        )
        .otherwise(None)
        .alias("expected_rotation_drtg")

    )
)


# ============================================================
# 8. EXPECTED ACTIVE DEFENSIVE ROTATION
# ============================================================
#
# Same calculation as above, except players classified as
# unavailable are removed.
#
# pregame_available:
#
#     1 = expected available
#     0 = expected unavailable
#
# CURRENTLY:
# This may be your historical comment-based proxy.
#
# LATER:
# Replace with NBA daily-lineup rosterStatus.
# ============================================================

lf_active_defence = (
    lf_player_stats

    .filter(
        (pl.col("pregame_available") == 1)
        &
        pl.col("numMinutes_ewma").is_not_null()
        &
        pl.col("defensiveRating_ewma").is_not_null()
        &
        (pl.col("numMinutes_ewma") > 0)
    )

    .with_columns(

        (
            pl.col("defensiveRating_ewma")
            * pl.col("numMinutes_ewma")
        )
        .alias("active_drtg_component")

    )

    .group_by([
        "season",
        "gameId",
        "playerteamId",
    ])

    .agg([

        pl.col("active_drtg_component")
        .sum()
        .alias("active_drtg_sum"),


        pl.col("numMinutes_ewma")
        .sum()
        .alias("active_expected_minutes"),


        pl.col("personId")
        .n_unique()
        .alias("expected_active_rotation_players"),

    ])

    .with_columns(

        pl.when(
            pl.col("active_expected_minutes") > 0
        )
        .then(
            pl.col("active_drtg_sum")
            / pl.col("active_expected_minutes")
        )
        .otherwise(None)
        .alias("expected_active_rotation_drtg")

    )
)


# ============================================================
# 9. JOIN EXPECTED ROTATION DRtg TO PLAYER DATA
# ============================================================
# Required before calculating individual defensive impact lost.
# ============================================================

lf_player_stats = (
    lf_player_stats

    .join(

        lf_rotation_defence.select([
            "season",
            "gameId",
            "playerteamId",
            "expected_rotation_drtg",
        ]),

        on=[
            "season",
            "gameId",
            "playerteamId",
        ],

        how="left",
    )
)


# ============================================================
# 10. DEFENSIVE IMPACT LOST — PLAYER LEVEL
# ============================================================
#
# Lower defensive rating = better defensive performance.
#
# Therefore:
#
# team DRtg - player DRtg
#
# is positive when the player has a better historical
# defensive rating than the expected team rotation.
#
#
# Example:
#
# expected_rotation_drtg = 113
# Gobert DRtg            = 104
# expected minutes       = 34
#
# 34 × (113 - 104)
# = 306
#
# Losing him therefore generates a large positive value.
#
# A poor defender can generate a negative value.
# ============================================================

lf_player_stats = lf_player_stats.with_columns(

    pl.when(
        (pl.col("pregame_available") == 0)
        &
        pl.col("numMinutes_ewma").is_not_null()
        &
        pl.col("defensiveRating_ewma").is_not_null()
        &
        pl.col("expected_rotation_drtg").is_not_null()
    )

    .then(
        pl.col("numMinutes_ewma")
        *
        (
            pl.col("expected_rotation_drtg")
            - pl.col("defensiveRating_ewma")
        )
    )

    .otherwise(0.0)

    .alias("defensive_impact_lost_component")

)


# ============================================================
# 11. AGGREGATE DEFENSIVE ABSENCES
# ============================================================

lf_defensive_absences = (
    lf_player_stats

    .group_by([
        "season",
        "gameId",
        "playerteamId",
    ])

    .agg([

        # ----------------------------------------------------
        # Total defensive quality lost
        # ----------------------------------------------------

        pl.col("defensive_impact_lost_component")
        .sum()
        .alias("defensive_impact_lost"),


        # ----------------------------------------------------
        # Number of meaningful rotation players unavailable
        #
        # 15 expected minutes is currently the threshold.
        # ----------------------------------------------------

        (
            (
                (pl.col("pregame_available") == 0)
                &
                (pl.col("numMinutes_ewma") >= 15)
            )
            .cast(pl.Int8)
            .sum()
        )
        .alias("defensive_rotation_players_out"),


        # ----------------------------------------------------
        # Total expected minutes unavailable
        # ----------------------------------------------------

        pl.when(
            pl.col("pregame_available") == 0
        )
        .then(
            pl.col("numMinutes_ewma")
            .fill_null(0)
        )
        .otherwise(0.0)
        .sum()
        .alias("defensive_minutes_out"),

    ])
)


# ============================================================
# 12. BUILD TEAM DEFENSIVE PROFILE
# ============================================================

lf_team_defensive_profile = (
    lf_team_game

    .join(
        lf_rotation_defence,

        on=[
            "season",
            "gameId",
            "playerteamId",
        ],

        how="left",
    )

    .join(
        lf_active_defence,

        on=[
            "season",
            "gameId",
            "playerteamId",
        ],

        how="left",
    )

    .join(
        lf_defensive_absences,

        on=[
            "season",
            "gameId",
            "playerteamId",
        ],

        how="left",
    )
)


# ============================================================
# 13. DEFENSIVE ROTATION CHANGE
# ============================================================
#
# This is one of the most useful injury-adjusted features.
#
# Positive:
# available rotation is WORSE defensively
#
# Negative:
# available rotation is BETTER defensively
#
# Example:
#
# Full expected rotation = 109
# Active rotation        = 113
#
# change = +4
#
# Defence is expected to deteriorate because of absences.
# ============================================================

lf_team_defensive_profile = (
    lf_team_defensive_profile

    .with_columns(

        (
            pl.col("expected_active_rotation_drtg")
            - pl.col("expected_rotation_drtg")
        )
        .alias("rotation_drtg_change")

    )
)


# ============================================================
# 14. CREATE OPPONENT DEFENSIVE PROFILE
# ============================================================
#
# Convert the defensive profile belonging to the opponent into
# features that can be attached to each offensive player.
# ============================================================

lf_opponent_defensive_profile = (
    lf_team_defensive_profile

    .select([

        "season",
        "gameId",

        pl.col("playerteamId")
        .alias("opponentteamId"),


        # ----------------------------------------------------
        # Historical team defence
        # ----------------------------------------------------

        pl.col("points_allowed_lag1")
        .alias("opp_points_allowed_lag1"),

        pl.col("points_allowed_ewma")
        .alias("opp_points_allowed_ewma"),

        pl.col("points_allowed_avg5")
        .alias("opp_points_allowed_avg5"),

        pl.col("points_allowed_avg10")
        .alias("opp_points_allowed_avg10"),

        pl.col("points_allowed_avg20")
        .alias("opp_points_allowed_avg20"),

        pl.col("points_allowed_trend_5v20")
        .alias("opp_points_allowed_trend_5v20"),


        # ----------------------------------------------------
        # Expected rotation defence
        # ----------------------------------------------------

        pl.col("expected_rotation_drtg")
        .alias("opp_expected_rotation_drtg"),

        pl.col("expected_active_rotation_drtg")
        .alias("opp_expected_active_rotation_drtg"),

        pl.col("rotation_drtg_change")
        .alias("opp_rotation_drtg_change"),


        # ----------------------------------------------------
        # Defensive absences
        # ----------------------------------------------------

        pl.col("defensive_impact_lost")
        .alias("opp_defensive_impact_lost"),

        pl.col("defensive_rotation_players_out")
        .alias("opp_defensive_rotation_players_out"),

        pl.col("defensive_minutes_out")
        .alias("opp_defensive_minutes_out"),


        # ----------------------------------------------------
        # Rotation context
        # ----------------------------------------------------

        pl.col("rotation_player_count")
        .alias("opp_rotation_player_count"),

        pl.col("expected_active_rotation_players")
        .alias("opp_expected_active_rotation_players"),

    ])
)


# ============================================================
# 15. JOIN OPPONENT DEFENSIVE PROFILE TO PLAYER DATA
# ============================================================

lf_player_stats = (
    lf_player_stats

    .join(
        lf_opponent_defensive_profile,

        on=[
            "season",
            "gameId",
            "opponentteamId",
        ],

        how="left",
    )
)

### 1.4 Final feature assembly
Select offensive and defensive features and create the modelling LazyFrame.

In [ ]:
# ============================================================
# POINTS MODEL FEATURE SET
# ============================================================

TARGET = "points"


# ============================================================
# PLAYER OFFENSIVE / FORM FEATURES
# ============================================================

PLAYER_FEATURES = [

    # --------------------------------------------------------
    # Minutes / Playing Opportunity
    # --------------------------------------------------------
    "numMinutes_lag1",
    "numMinutes_ewma",
    "numMinutes_avg5",
    "numMinutes_avg10",
    "minutes_trend_5v10",

    # --------------------------------------------------------
    # Points Form
    # --------------------------------------------------------
    "points_lag1",
    "points_ewma",
    "points_avg5",
    "points_avg10",
    "points_trend_5v_ewma",

    # --------------------------------------------------------
    # Field Goals
    # --------------------------------------------------------
    "fieldGoalsAttempted_lag1",
    "fieldGoalsAttempted_ewma",
    "fieldGoalsAttempted_avg5",
    "fieldGoalsAttempted_avg10",

    "fieldGoalsMade_lag1",
    "fieldGoalsMade_ewma",
    "fieldGoalsMade_avg5",
    "fieldGoalsMade_avg10",

    # --------------------------------------------------------
    # Three Pointers
    # --------------------------------------------------------
    "threePointersAttempted_lag1",
    "threePointersAttempted_ewma",
    "threePointersAttempted_avg5",
    "threePointersAttempted_avg10",

    "threePointersMade_lag1",
    "threePointersMade_ewma",
    "threePointersMade_avg5",
    "threePointersMade_avg10",

    # --------------------------------------------------------
    # Free Throws
    # --------------------------------------------------------
    "freeThrowsAttempted_lag1",
    "freeThrowsAttempted_ewma",
    "freeThrowsAttempted_avg5",
    "freeThrowsAttempted_avg10",

    "freeThrowsMade_lag1",
    "freeThrowsMade_ewma",
    "freeThrowsMade_avg5",
    "freeThrowsMade_avg10",

    # --------------------------------------------------------
    # Usage
    # --------------------------------------------------------
    "usagePercentage_lag1",
    "usagePercentage_ewma",
    "usagePercentage_avg5",
    "usagePercentage_avg10",
    "usage_trend_5v10",

    # --------------------------------------------------------
    # Pace
    # --------------------------------------------------------
    "pace_lag1",
    "pace_ewma",
    "pace_avg5",
    "pace_avg10",

    # --------------------------------------------------------
    # Opportunity Trends
    # --------------------------------------------------------
    "fga_trend_5v10",
    "threepa_trend_5v10",
    "fta_trend_5v10",
]


# ============================================================
# OPPONENT DEFENSIVE PROFILE
# ============================================================

DEFENSIVE_FEATURES = [

    # --------------------------------------------------------
    # Points Allowed
    # --------------------------------------------------------
    "opp_points_allowed_ewma",
    "opp_points_allowed_avg5",
    "opp_points_allowed_avg10",
    "opp_points_allowed_avg20",
    "opp_points_allowed_trend_5v20",

    # --------------------------------------------------------
    # Expected Defensive Rotation
    # --------------------------------------------------------
    "opp_expected_rotation_drtg",

    # --------------------------------------------------------
    # Current Expected Active Defence
    # --------------------------------------------------------
    # These require genuine pre-game lineup/availability data.
    "opp_expected_active_rotation_drtg",

    # --------------------------------------------------------
    # Defensive Personnel Missing
    # --------------------------------------------------------
    "opp_defensive_impact_lost",
    "opp_defensive_rotation_players_out",
    "opp_defensive_minutes_out",
]


# ============================================================
# COMBINE MODEL FEATURES
# ============================================================

POINTS_FEATURES = (
    PLAYER_FEATURES
    + DEFENSIVE_FEATURES
)


# ============================================================
# IDENTIFIER / CONTEXT COLUMNS
# ============================================================

ID_COLS = [
    "personId",
    "numMinutes",
    "gameId",
    "gameDateTimeEst",
    "season",
    "playerteamId",
    "opponentteamId",
    "home",
    "is_starter",
    "pregame_available"
]


# ============================================================
# BUILD POINTS MODEL LAZYFRAME
# ============================================================

lf_player_stats_points = (
    lf_player_stats

    # Player must actually have played
    .filter(
        pl.col("points").is_not_null()
        &
        pl.col("numMinutes").is_not_null()
        &
        (pl.col("numMinutes") > 0)
    )

    .select(
        ID_COLS
        + [TARGET]
        + POINTS_FEATURES
    )

    .sort([
        "gameDateTimeEst",
        "gameId",
        "personId"
    ])
)

### 1.5 Feature engineering sanity check
Materialise the prepared dataset for inspection. This may be memory-intensive on the full historical dataset.

In [ ]:
lf_player_stats_points.collect()

personId,numMinutes,gameId,gameDateTimeEst,season,playerteamId,opponentteamId,home,is_starter,pregame_available,points,numMinutes_lag1,numMinutes_ewma,numMinutes_avg5,numMinutes_avg10,minutes_trend_5v10,points_lag1,points_ewma,points_avg5,points_avg10,points_trend_5v_ewma,fieldGoalsAttempted_lag1,fieldGoalsAttempted_ewma,fieldGoalsAttempted_avg5,fieldGoalsAttempted_avg10,fieldGoalsMade_lag1,fieldGoalsMade_ewma,fieldGoalsMade_avg5,fieldGoalsMade_avg10,threePointersAttempted_lag1,threePointersAttempted_ewma,threePointersAttempted_avg5,threePointersAttempted_avg10,threePointersMade_lag1,threePointersMade_ewma,threePointersMade_avg5,threePointersMade_avg10,freeThrowsAttempted_lag1,freeThrowsAttempted_ewma,freeThrowsAttempted_avg5,freeThrowsAttempted_avg10,freeThrowsMade_lag1,freeThrowsMade_ewma,freeThrowsMade_avg5,freeThrowsMade_avg10,usagePercentage_lag1,usagePercentage_ewma,usagePercentage_avg5,usagePercentage_avg10,usage_trend_5v10,pace_lag1,pace_ewma,pace_avg5,pace_avg10,fga_trend_5v10,threepa_trend_5v10,fta_trend_5v10,opp_points_allowed_ewma,opp_points_allowed_avg5,opp_points_allowed_avg10,opp_points_allowed_avg20,opp_points_allowed_trend_5v20,opp_expected_rotation_drtg,opp_expected_active_rotation_drtg,opp_defensive_impact_lost,opp_defensive_rotation_players_out,opp_defensive_minutes_out
i64,f64,i64,date,i32,i64,i64,i64,i8,i8,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i64,f64
23,39.0,29600001,1996-11-01,1996,1610612741,1610612738,0,1,1,13.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,0.0,0,0.0
26,11.0,29600001,1996-11-01,1996,1610612741,1610612738,0,1,1,4.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,0.0,0,0.0
54,4.0,29600001,1996-11-01,1996,1610612741,1610612738,0,0,1,1.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,0.0,0,0.0
70,20.0,29600001,1996-11-01,1996,1610612741,1610612738,0,1,1,0.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,0.0,0,0.0
82,10.0,29600001,1996-11-01,1996,1610612741,1610612738,0,1,1,4.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,0.0,0,0.0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
1642345,26.22,52500211,2026-04-17,2025,1610612756,1610612744,1,1,1,10.0,23.17,20.266491,18.592,22.709,-4.117,3.0,3.388682,5.0,6.6,1.611318,2.0,2.551697,3.6,4.4,1.0,1.297476,2.2,3.0,0.0,8.1668e-27,0.0,0.0,0.0,0.0,0.0,0.0,2.0,1.600143,1.2,1.6,1.0,0.79373,0.6,0.6,0.094,0.104603,0.1274,0.1163,0.0111,93.8,94.850866,96.85,100.15,-0.8,0.0,-0.4,119.497538,116.8,118.8,119.45,-2.65,120.773166,120.773166,0.0,0,0.0
1642346,6.35,52500211,2026-04-17,2025,1610612756,1610612744,1,0,1,0.0,9.1,15.080002,17.432,18.145,-0.713,2.0,6.126749,4.6,6.2,-1.526749,2.0,4.267351,3.8,4.8,1.0,2.534282,1.8,2.6,0.0

---
## 2. Model development and benchmarking
Explore training windows and candidate regression algorithms before choosing a mean-prediction model.

### 2.1 Optional environment setup
Install H2O if needed. Dependency-installation commands are environment-specific and may be skipped when packages are already available.

In [ ]:
!pip install h2o

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 266.3/266.3 MB 7.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for h2o: filename=h2o-3.46.0.12-py2.py3-none-any.whl size=266411237 sha256=af4dccaa4714bee5bac6f5a8f98ea7c5f2ae32c623e45e4161e9cb64e2d02f31
  Stored in directory: /root/.cache/pip/wheels/2d/d1/1c/742fac210b3d4c1578163533b3f96a51659aac2702838347af
Successfully built h2o


In [ ]:
# ============================================================
# Import necessary ML libraries - AutoML, SkLearn
# ============================================================


import polars as pl
import numpy as np

# -----------------------------------------------------------
# sklearn
# -----------------------------------------------------------
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
# -----------------------------------------------------------
# Other Boosting Models
# -----------------------------------------------------------
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

# -----------------------------------------------------------
# H2O AutoML
# -----------------------------------------------------------
import h2o
from h2o.automl import H2OAutoML


### 2.2 H2O and initial model setup
Initialise exploratory AutoML tools and prepare the features used in the benchmark experiments.

In [ ]:
# ============================================================
# INITIALISE H2O
# ============================================================

import h2o
from h2o.automl import H2OAutoML

h2o.init()


# ============================================================
# TARGET
# ============================================================

TARGET = "points"


# ============================================================
# MODEL FEATURES
# ============================================================

FEATURES = [

    # --------------------------------------------------------
    # MINUTES / PLAYING OPPORTUNITY
    # --------------------------------------------------------

    "numMinutes_lag1",
    "numMinutes_ewma",
    "numMinutes_avg5",
    "numMinutes_avg10",
    "minutes_trend_5v10",


    # --------------------------------------------------------
    # POINTS FORM
    # --------------------------------------------------------

    "points_lag1",
    "points_ewma",
    "points_avg5",
    "points_avg10",
    "points_trend_5v_ewma",


    # --------------------------------------------------------
    # FIELD GOALS
    # --------------------------------------------------------

    "fieldGoalsAttempted_lag1",
    "fieldGoalsAttempted_ewma",
    "fieldGoalsAttempted_avg5",
    "fieldGoalsAttempted_avg10",

    "fieldGoalsMade_lag1",
    "fieldGoalsMade_ewma",
    "fieldGoalsMade_avg5",
    "fieldGoalsMade_avg10",


    # --------------------------------------------------------
    # THREE POINTERS
    # --------------------------------------------------------

    "threePointersAttempted_lag1",
    "threePointersAttempted_ewma",
    "threePointersAttempted_avg5",
    "threePointersAttempted_avg10",

    "threePointersMade_lag1",
    "threePointersMade_ewma",
    "threePointersMade_avg5",
    "threePointersMade_avg10",


    # --------------------------------------------------------
    # FREE THROWS
    # --------------------------------------------------------

    "freeThrowsAttempted_lag1",
    "freeThrowsAttempted_ewma",
    "freeThrowsAttempted_avg5",
    "freeThrowsAttempted_avg10",

    "freeThrowsMade_lag1",
    "freeThrowsMade_ewma",
    "freeThrowsMade_avg5",
    "freeThrowsMade_avg10",


    # --------------------------------------------------------
    # USAGE
    # --------------------------------------------------------

    "usagePercentage_lag1",
    "usagePercentage_ewma",
    "usagePercentage_avg5",
    "usagePercentage_avg10",
    "usage_trend_5v10",


    # --------------------------------------------------------
    # PACE
    # --------------------------------------------------------

    "pace_lag1",
    "pace_ewma",
    "pace_avg5",
    "pace_avg10",


    # --------------------------------------------------------
    # SCORING / OPPORTUNITY TRENDS
    # --------------------------------------------------------

    "fga_trend_5v10",
    "threepa_trend_5v10",
    "fta_trend_5v10",


    # --------------------------------------------------------
    # OPPONENT HISTORICAL DEFENCE
    # --------------------------------------------------------

    "opp_points_allowed_ewma",
    "opp_points_allowed_avg5",
    "opp_points_allowed_avg10",
    "opp_points_allowed_avg20",
    "opp_points_allowed_trend_5v20",


    # --------------------------------------------------------
    # OPPONENT EXPECTED DEFENSIVE ROTATION
    # --------------------------------------------------------

    "opp_expected_rotation_drtg",
    "opp_expected_active_rotation_drtg",

    # Change caused by unavailable defenders
    #"opp_rotation_drtg_change",


    # --------------------------------------------------------
    # OPPONENT DEFENSIVE ABSENCES
    # --------------------------------------------------------

    "opp_defensive_impact_lost",
    "opp_defensive_rotation_players_out",
    "opp_defensive_minutes_out",
]


# ============================================================
# COLLECT POLARS LAZYFRAME
# ============================================================

df_points = (
    lf_player_stats_points
    .select(
        ["personId","gameId","gameDateTimeEst", "numMinutes", TARGET] + FEATURES
    )
    .collect()
)


# ============================================================
# REMOVE ROWS WITHOUT TARGET
# ============================================================

df_points = df_points.filter(
    pl.col(TARGET).is_not_null()
)

Checking whether there is an H2O instance running at http://localhost:54321..... not found.
Attempting to start a local H2O server...
  Java Version: [0.024s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate; [0.024s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate; openjdk version "21.0.12.1" 2026-08-18; OpenJDK Runtime Environment (build 21.0.12.1+1-1-24.04.4-Ubuntu); OpenJDK 64-Bit Server VM (build 21.0.12.1+1-1-24.04.4-Ubuntu, mixed mode, sharing)
  Starting server from /usr/local/lib/python3.13/dist-packages/h2o/backend/bin/h2o.jar
  Ice root: /tmp/tmp5qird2wf
  JVM stdout: /tmp/tmp5qird2wf/h2o_unknownUser_started_from_python.out
  JVM stderr: /tmp/tmp5qird2wf/h2o_unknownUser_started_from_python.err
  Server is running at http://127.0.0.1:54321
Connecting to H2O server a

H2O_cluster_uptime:,03 secs
H2O_cluster_timezone:,Etc/UTC
H2O_data_parsing_timezone:,UTC
H2O_cluster_version:,3.46.0.12
H2O_cluster_version_age:,2 months
H2O_cluster_name:,H2O_from_python_unknownUser_7polq2
H2O_cluster_total_nodes:,1
H2O_cluster_free_memory:,3.149 Gb
H2O_cluster_total_cores:,2
H2O_cluster_allowed_cores:,2
H2O_cluster_status:,"locked, healthy"



+------------------------------------------------------------------+
| You are running the community edition of H2O-3 OSS.              |
|                                                                  |
| For commercial use, H2O-3 Secure is now recommended.             |
| This includes production support, CVE fixes, multi-node scaling, |
| model artifact extraction, and more.                             |
| See h2o.ai/h2o-3/oss-vs-secure for additional details.           |
| Contact enterprise@h2o.ai to upgrade.                            |
+------------------------------------------------------------------+


### 2.3 Chronological training-window design
Define fixed validation and test periods, then compare alternative lengths of historical training data. The test period should remain held out during model selection.

In [ ]:
# ============================================================
# TRAINING WINDOW EXPERIMENT
#
# Goal:
# Compare different amounts of historical training data while
# keeping validation and test periods identical.
# ============================================================

from datetime import timedelta
import polars as pl


# ============================================================
# 1. FIND DATASET END DATE
# ============================================================

max_date = df_points["gameDateTimeEst"].max()

print(f"Dataset end date: {max_date}")


# ============================================================
# 2. DEFINE FIXED VALIDATION / TEST PERIODS
# ============================================================

# Most recent ~1 year = TEST
test_start_date = max_date - timedelta(days=365)

# Previous ~1 year = VALIDATION
validation_start_date = test_start_date - timedelta(days=365)


print("\n" + "=" * 70)
print("FIXED EVALUATION PERIODS")
print("=" * 70)

print(f"Validation starts: {validation_start_date}")
print(f"Test starts:       {test_start_date}")
print(f"Dataset ends:      {max_date}")


# ============================================================
# 3. FIXED VALIDATION SET
# ============================================================

validation_df = df_points.filter(
    (pl.col("gameDateTimeEst") >= validation_start_date)
    & (pl.col("gameDateTimeEst") < test_start_date)
)


# ============================================================
# 4. FIXED TEST SET
# ============================================================

test_df = df_points.filter(
    pl.col("gameDateTimeEst") >= test_start_date
)


# ============================================================
# 5. CREATE TRAINING WINDOW FUNCTION
# ============================================================

def create_training_window(df, years):

    training_start_date = (
        validation_start_date
        - timedelta(days=365.25 * years)
    )

    train_df = df.filter(
        (pl.col("gameDateTimeEst") >= training_start_date)
        & (pl.col("gameDateTimeEst") < validation_start_date)
    )

    return train_df


# ============================================================
# 6. CREATE DIFFERENT TRAINING WINDOWS
# ============================================================

train_5yr = create_training_window(
    df_points,
    5
)

train_10yr = create_training_window(
    df_points,
    10
)

train_15yr = create_training_window(
    df_points,
    15
)


# All available historical data before validation
train_all = df_points.filter(
    pl.col("gameDateTimeEst") < validation_start_date
)


# ============================================================
# 7. VALIDATION / TEST MATRICES
# ============================================================

X_validation = validation_df.select(FEATURES).to_pandas()
y_validation = validation_df[TARGET].to_numpy()

X_test = test_df.select(FEATURES).to_pandas()
y_test = test_df[TARGET].to_numpy()


# ============================================================
# 8. CHECK WINDOWS
# ============================================================

training_sets = {
    "5 Years": train_5yr,
    "10 Years": train_10yr,
    "15 Years": train_15yr,
    "All History": train_all
}


print("\n" + "=" * 70)
print("TRAINING WINDOWS")
print("=" * 70)


for name, df in training_sets.items():

    print(f"\n{name}")

    print(
        f"Dates: "
        f"{df['gameDateTimeEst'].min()} "
        f"→ "
        f"{df['gameDateTimeEst'].max()}"
    )

    print(
        f"Observations: {len(df):,}"
    )


print("\nValidation:")
print(
    validation_df["gameDateTimeEst"].min(),
    "→",
    validation_df["gameDateTimeEst"].max()
)

print(f"Observations: {len(validation_df):,}")


print("\nTest:")
print(
    test_df["gameDateTimeEst"].min(),
    "→",
    test_df["gameDateTimeEst"].max()
)

print(f"Observations: {len(test_df):,}")

print("=" * 70)

Dataset end date: 2026-04-17

FIXED EVALUATION PERIODS
Validation starts: 2024-04-17
Test starts:       2025-04-17
Dataset ends:      2026-04-17

TRAINING WINDOWS

5 Years
Dates: 2019-10-22 → 2024-04-16
Observations: 123,095

10 Years
Dates: 2014-10-28 → 2024-04-16
Observations: 252,387

15 Years
Dates: 2009-10-27 → 2024-04-16
Observations: 373,206

All History
Dates: 1996-11-01 → 2024-04-16
Observations: 676,929

Validation:
2024-04-17 → 2025-04-16
Observations: 26,324

Test:
2025-04-18 → 2026-04-17
Observations: 26,833


### 2.4 Baseline evaluation
Inspect the initial holdout metrics and establish a comparison point for subsequent model experiments.

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np


y_test = test_df["points"].to_numpy()


# ============================================================
# EWMA POINTS BASELINE
# ============================================================

baseline_pred = (
    test_df["points_ewma"]
    .to_numpy()
)

mask = ~np.isnan(baseline_pred)

baseline_mae = mean_absolute_error(
    y_test[mask],
    baseline_pred[mask]
)

baseline_rmse = np.sqrt(
    mean_squared_error(
        y_test[mask],
        baseline_pred[mask]
    )
)

baseline_r2 = r2_score(
    y_test[mask],
    baseline_pred[mask]
)


print("EWMA Points Baseline")
print("--------------------")
print(f"MAE:  {baseline_mae:.3f}")
print(f"RMSE: {baseline_rmse:.3f}")
print(f"R²:   {baseline_r2:.3f}")

EWMA Points Baseline
--------------------
MAE:  5.106
RMSE: 6.757
R²:   0.391


### 2.5 Multi-model validation experiments
Train candidate estimators across training windows and collect comparable validation predictions.

In [ ]:
import time
from datetime import datetime

from sklearn.ensemble import (
    HistGradientBoostingRegressor,
    RandomForestRegressor
)

from sklearn.svm import SVR
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor


# ============================================================
# HELPER FUNCTIONS FOR TRAINING STATUS
# ============================================================

def print_training_start(model_name, training_window):
    print("\n" + "=" * 70)
    print(f"STARTING: {model_name}")
    print(f"TRAINING WINDOW: {training_window}")
    print(f"Time: {datetime.now().strftime('%H:%M:%S')}")
    print("=" * 70)


def print_training_end(model_name, training_window, start_time):
    elapsed = time.time() - start_time

    print("-" * 70)
    print(f"COMPLETED: {model_name}")
    print(f"TRAINING WINDOW: {training_window}")
    print(f"Training time: {elapsed:.2f} seconds")
    print(f"Time: {datetime.now().strftime('%H:%M:%S')}")
    print("=" * 70 + "\n")


# ============================================================
# TRAINING DATASETS
# ============================================================

training_sets = {
    "5_years": train_5yr,
    "10_years": train_10yr,
    "15_years": train_15yr,
    "all_history": train_all
}


# ============================================================
# STORAGE
#
# trained_models["5_years"]["HGB"]
# validation_predictions["5_years"]["HGB"]
# ============================================================

trained_models = {}
validation_predictions = {}


# ============================================================
# TRAIN ALL MODELS ACROSS ALL TRAINING WINDOWS
# ============================================================

total_start_time = time.time()


for window_name, train_data in training_sets.items():

    print("\n")
    print("#" * 70)
    print(f"TRAINING WINDOW: {window_name.upper()}")
    print("#" * 70)

    print(
        f"Date range: "
        f"{train_data['gameDateTimeEst'].min()} "
        f"→ "
        f"{train_data['gameDateTimeEst'].max()}"
    )

    print(f"Training observations: {len(train_data):,}")


    # ========================================================
    # CREATE TRAINING MATRICES
    # ========================================================

    X_train = (
        train_data
        .select(FEATURES)
        .to_pandas()
    )

    y_train = (
        train_data[TARGET]
        .to_numpy()
    )


    # ========================================================
    # STORAGE FOR CURRENT WINDOW
    # ========================================================

    trained_models[window_name] = {}
    validation_predictions[window_name] = {}


    # ========================================================
    # 1. HISTGRADIENTBOOSTINGREGRESSOR
    # ========================================================

    model_name = "HGB"

    print_training_start(
        "HistGradientBoostingRegressor",
        window_name
    )

    start_time = time.time()


    model = HistGradientBoostingRegressor(
        learning_rate=0.03,
        max_iter=500,
        max_leaf_nodes=127,
        l2_regularization=0.5,
        max_depth=4,
        min_samples_leaf=75,
        max_bins=127,
        random_state=42
    )


    model.fit(
        X_train,
        y_train
    )


    pred = model.predict(
        X_validation
    )


    trained_models[window_name][model_name] = model

    validation_predictions[window_name][model_name] = pred


    print_training_end(
        "HistGradientBoostingRegressor",
        window_name,
        start_time
    )

"""
    # ========================================================
    # 2. RANDOM FOREST REGRESSOR
    # ========================================================

    model_name = "RF"

    print_training_start(
        "RandomForestRegressor",
        window_name
    )

    start_time = time.time()


    model = RandomForestRegressor(
        n_estimators=100,
        max_depth=10,
        random_state=42,
        n_jobs=-1
    )


    model.fit(
        X_train,
        y_train
    )


    pred = model.predict(
        X_validation
    )


    trained_models[window_name][model_name] = model

    validation_predictions[window_name][model_name] = pred


    print_training_end(
        "RandomForestRegressor",
        window_name,
        start_time
    )
"""

    # ========================================================
    # 3. XGBOOST REGRESSOR
    # ========================================================

    model_name = "XGB"

    print_training_start(
        "XGBRegressor",
        window_name
    )

    start_time = time.time()


    model = XGBRegressor(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=5,
        min_child_weight=5,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.1,
        reg_lambda=1.0,
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1
    )


    model.fit(
        X_train,
        y_train
    )


    pred = model.predict(
        X_validation
    )


    trained_models[window_name][model_name] = model

    validation_predictions[window_name][model_name] = pred


    print_training_end(
        "XGBRegressor",
        window_name,
        start_time
    )


    # ========================================================
    # 4. LIGHTGBM REGRESSOR
    # ========================================================

    model_name = "LGBM"

    print_training_start(
        "LGBMRegressor",
        window_name
    )

    start_time = time.time()


    model = LGBMRegressor(
        n_estimators=500,
        learning_rate=0.05,
        num_leaves=31,
        max_depth=-1,
        min_child_samples=30,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.1,
        reg_lambda=1.0,
        random_state=42,
        n_jobs=-1,
        verbosity=-1
    )


    model.fit(
        X_train,
        y_train
    )


    pred = model.predict(
        X_validation
    )


    trained_models[window_name][model_name] = model

    validation_predictions[window_name][model_name] = pred


    print_training_end(
        "LGBMRegressor",
        window_name,
        start_time
    )


# ========================================================
# 5. LINEAR SVR REGRESSOR
# ========================================================

model_name = "LinearSVR"

print_training_start(
    "LinearSVR",
    window_name
)

start_time = time.time()


# --------------------------------------------------------
# CREATE MODEL PIPELINE
#
# StandardScaler is important because LinearSVR is
# sensitive to differences in feature scale.
# --------------------------------------------------------

model = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),
    (
        "svr",
        LinearSVR(
            C=1.0,
            epsilon=0.5,
            max_iter=10000,
            random_state=42
        )
    )
])


# --------------------------------------------------------
# TRAIN MODEL
# --------------------------------------------------------

model.fit(
    X_train,
    y_train
)


# --------------------------------------------------------
# VALIDATION PREDICTIONS
# --------------------------------------------------------

pred = model.predict(
    X_validation
)


# --------------------------------------------------------
# STORE MODEL AND PREDICTIONS
# --------------------------------------------------------

trained_models[window_name][model_name] = model

validation_predictions[window_name][model_name] = pred


print_training_end(
    "LinearSVR",
    window_name,
    start_time
)


# ============================================================
# ALL MODELS COMPLETE
# ============================================================

total_elapsed = time.time() - total_start_time

print("\n" + "=" * 70)
print("ALL MODELS HAVE FINISHED TRAINING")
print("=" * 70)

print(f"Training windows tested: {len(training_sets)}")
print("Models per window:       4")
print(f"Total model fits:         {len(training_sets) * 4}")

print("\nTraining windows:")
for window_name in training_sets:
    print(f"  - {window_name}")

print("\nValidation predictions generated for:")
print("  - HGB")
print("  - RF")
print("  - XGB")
print("  - LGBM")

print(
    f"\nTotal training time: "
    f"{total_elapsed / 60:.2f} minutes"
)

print(
    f"Completed at: "
    f"{datetime.now().strftime('%H:%M:%S')}"
)

print("=" * 70)



######################################################################
TRAINING WINDOW: 5_YEARS
######################################################################
Date range: 2019-10-22 → 2024-04-16
Training observations: 123,095

STARTING: HistGradientBoostingRegressor
TRAINING WINDOW: 5_years
Time: 09:07:09
----------------------------------------------------------------------
COMPLETED: HistGradientBoostingRegressor
TRAINING WINDOW: 5_years
Training time: 10.71 seconds
Time: 09:07:19


STARTING: RandomForestRegressor
TRAINING WINDOW: 5_years
Time: 09:07:19
----------------------------------------------------------------------
COMPLETED: RandomForestRegressor
TRAINING WINDOW: 5_years
Training time: 517.86 seconds
Time: 09:15:57


STARTING: XGBRegressor
TRAINING WINDOW: 5_years
Time: 09:15:57
----------------------------------------------------------------------
COMPLETED: XGBRegressor
TRAINING WINDOW: 5_years
Training time: 19.27 seconds
Time: 09:16:17


STARTING: LGBMRegressor

### 2.6 Validation model selection
Summarise validation performance and identify the leading candidate models.

In [ ]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

import pandas as pd
import numpy as np


# ============================================================
# COMPARE VALIDATION PERFORMANCE
# ============================================================

results = []


for window_name, model_predictions in validation_predictions.items():

    for model_name, pred in model_predictions.items():

        mae = mean_absolute_error(
            y_validation,
            pred
        )

        rmse = np.sqrt(
            mean_squared_error(
                y_validation,
                pred
            )
        )

        r2 = r2_score(
            y_validation,
            pred
        )

        bias = np.mean(
            y_validation - pred
        )


        results.append({
            "Training Window": window_name,
            "Model": model_name,
            "MAE": mae,
            "RMSE": rmse,
            "R2": r2,
            "Bias": bias
        })


# ============================================================
# RESULTS TABLE
# ============================================================

results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    "MAE",
    ascending=True
).reset_index(drop=True)


print("\n" + "=" * 80)
print("VALIDATION MODEL COMPARISON")
print("=" * 80)

print(
    results_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.3f}"
    )
)

print("=" * 80)


# ============================================================
# BEST MODEL
# ============================================================

best_result = results_df.iloc[0]

print("\nBEST VALIDATION RESULT")
print("-" * 40)

print(
    f"Training Window: "
    f"{best_result['Training Window']}"
)

print(
    f"Model:           "
    f"{best_result['Model']}"
)

print(
    f"MAE:             "
    f"{best_result['MAE']:.3f}"
)

print(
    f"RMSE:            "
    f"{best_result['RMSE']:.3f}"
)

print(
    f"R²:              "
    f"{best_result['R2']:.3f}"
)

print(
    f"Bias:            "
    f"{best_result['Bias']:+.3f}"
)


VALIDATION MODEL COMPARISON
Training Window Model   MAE  RMSE    R2   Bias
       15_years  LGBM 4.622 6.022 0.532 -0.017
    all_history   XGB 4.622 6.023 0.532 -0.029
    all_history  LGBM 4.623 6.021 0.532 -0.025
       15_years   XGB 4.623 6.024 0.532 -0.021
       10_years   XGB 4.625 6.030 0.531 -0.012
       15_years    RF 4.625 6.029 0.531  0.049
       10_years  LGBM 4.626 6.030 0.531 -0.009
       10_years    RF 4.626 6.031 0.531  0.046
       15_years   HGB 4.626 6.024 0.532 -0.023
        5_years   XGB 4.627 6.038 0.530  0.003
       10_years   HGB 4.627 6.027 0.532 -0.017
    all_history   HGB 4.628 6.024 0.532 -0.038
        5_years   HGB 4.628 6.027 0.532 -0.008
        5_years  LGBM 4.628 6.036 0.530 -0.004
    all_history    RF 4.631 6.038 0.530  0.061
        5_years    RF 4.636 6.040 0.530  0.006

BEST VALIDATION RESULT
----------------------------------------
Training Window: 15_years
Model:           LGBM
MAE:             4.622
RMSE:            6.022
R²:          

### 2.7 Final chronological test comparison
Refit selected models using the designated pre-test data and compare performance on the fixed test period. This section documents the exploratory V1 benchmark.

In [ ]:
import time
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
import polars as pl

from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor


# ============================================================
# FINAL TESTING EXPERIMENT
#
# Models:
#   - HGB
#   - XGB
#   - LGBM
#
# Training windows:
#   - 10 years
#   - 15 years
#   - All history
#
# Validation data is now included in final training data.
# Test set remains untouched until prediction.
# ============================================================


# ============================================================
# HELPER FUNCTIONS
# ============================================================

def print_training_start(model_name, training_window):
    print("\n" + "=" * 70)
    print(f"STARTING: {model_name}")
    print(f"TRAINING WINDOW: {training_window}")
    print(f"Time: {datetime.now().strftime('%H:%M:%S')}")
    print("=" * 70)


def print_training_end(model_name, training_window, start_time):
    elapsed = time.time() - start_time

    print("-" * 70)
    print(f"COMPLETED: {model_name}")
    print(f"TRAINING WINDOW: {training_window}")
    print(f"Training time: {elapsed:.2f} seconds")
    print(f"Time: {datetime.now().strftime('%H:%M:%S')}")
    print("=" * 70 + "\n")


# ============================================================
# 1. CREATE FINAL TRAINING WINDOWS
#
# IMPORTANT:
# Windows now end at test_start_date rather than
# validation_start_date.
# ============================================================

def create_final_training_window(df, years):

    training_start_date = (
        test_start_date
        - timedelta(days=365.25 * years)
    )

    return df.filter(
        (pl.col("gameDateTimeEst") >= training_start_date)
        & (pl.col("gameDateTimeEst") < test_start_date)
    )


final_train_10yr = create_final_training_window(
    df_points,
    10
)

final_train_15yr = create_final_training_window(
    df_points,
    15
)

final_train_all = df_points.filter(
    pl.col("gameDateTimeEst") < test_start_date
)


# ============================================================
# 2. TRAINING DATASETS
# ============================================================

final_training_sets = {
    "10_years": final_train_10yr,
    "15_years": final_train_15yr,
    "all_history": final_train_all
}


# ============================================================
# 3. TEST DATA
# ============================================================

X_test = (
    test_df
    .select(FEATURES)
    .to_pandas()
)

y_test = (
    test_df[TARGET]
    .to_numpy()
)


# ============================================================
# 4. CHECK FINAL TRAINING WINDOWS
# ============================================================

print("\n" + "=" * 70)
print("FINAL TESTING DATASET")
print("=" * 70)

for window_name, train_data in final_training_sets.items():

    print(f"\n{window_name}")

    print(
        f"Training dates: "
        f"{train_data['gameDateTimeEst'].min()} "
        f"→ "
        f"{train_data['gameDateTimeEst'].max()}"
    )

    print(
        f"Training observations: "
        f"{len(train_data):,}"
    )


print("\nTEST SET")

print(
    f"Test dates: "
    f"{test_df['gameDateTimeEst'].min()} "
    f"→ "
    f"{test_df['gameDateTimeEst'].max()}"
)

print(
    f"Test observations: "
    f"{len(test_df):,}"
)

print("=" * 70)


# ============================================================
# 5. STORAGE
# ============================================================

final_models = {}
test_predictions = {}

total_start_time = time.time()


# ============================================================
# 6. TRAIN MODELS
# ============================================================

for window_name, train_data in final_training_sets.items():

    print("\n")
    print("#" * 70)
    print(f"FINAL TRAINING WINDOW: {window_name.upper()}")
    print("#" * 70)


    # --------------------------------------------------------
    # CREATE TRAINING MATRICES
    # --------------------------------------------------------

    X_train_final = (
        train_data
        .select(FEATURES)
        .to_pandas()
    )

    y_train_final = (
        train_data[TARGET]
        .to_numpy()
    )


    final_models[window_name] = {}
    test_predictions[window_name] = {}


    # ========================================================
    # 1. HISTGRADIENTBOOSTING
    # ========================================================

    model_name = "HGB"

    print_training_start(
        "HistGradientBoostingRegressor",
        window_name
    )

    start_time = time.time()


    model = HistGradientBoostingRegressor(
        learning_rate=0.03,
        max_iter=500,
        max_leaf_nodes=127,
        l2_regularization=0.5,
        max_depth=4,
        min_samples_leaf=75,
        max_bins=127,
        random_state=42
    )


    model.fit(
        X_train_final,
        y_train_final
    )


    pred = model.predict(
        X_test
    )


    final_models[window_name][model_name] = model
    test_predictions[window_name][model_name] = pred


    print_training_end(
        "HistGradientBoostingRegressor",
        window_name,
        start_time
    )


    # ========================================================
    # 2. XGBOOST
    # ========================================================

    model_name = "XGB"

    print_training_start(
        "XGBRegressor",
        window_name
    )

    start_time = time.time()


    model = XGBRegressor(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=5,
        min_child_weight=5,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.1,
        reg_lambda=1.0,
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1
    )


    model.fit(
        X_train_final,
        y_train_final
    )


    pred = model.predict(
        X_test
    )


    final_models[window_name][model_name] = model
    test_predictions[window_name][model_name] = pred


    print_training_end(
        "XGBRegressor",
        window_name,
        start_time
    )


    # ========================================================
    # 3. LIGHTGBM
    # ========================================================

    model_name = "LGBM"

    print_training_start(
        "LGBMRegressor",
        window_name
    )

    start_time = time.time()


    model = LGBMRegressor(
        n_estimators=500,
        learning_rate=0.05,
        num_leaves=31,
        max_depth=-1,
        min_child_samples=30,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.1,
        reg_lambda=1.0,
        random_state=42,
        n_jobs=-1,
        verbosity=-1
    )


    model.fit(
        X_train_final,
        y_train_final
    )


    pred = model.predict(
        X_test
    )


    final_models[window_name][model_name] = model
    test_predictions[window_name][model_name] = pred


    print_training_end(
        "LGBMRegressor",
        window_name,
        start_time
    )


# ============================================================
# 7. ALL FINAL MODELS COMPLETE
# ============================================================

total_elapsed = time.time() - total_start_time

print("\n" + "=" * 70)
print("ALL FINAL MODELS HAVE FINISHED TRAINING")
print("=" * 70)

print(f"Training windows tested: {len(final_training_sets)}")
print("Models per window:       3")
print("Total model fits:         9")

print(
    f"\nTotal training time: "
    f"{total_elapsed / 60:.2f} minutes"
)

print(
    f"Completed at: "
    f"{datetime.now().strftime('%H:%M:%S')}"
)

print("=" * 70)


# ============================================================
# 8. TEST SET EVALUATION
# ============================================================

results = []


for window_name, model_predictions in test_predictions.items():

    for model_name, pred in model_predictions.items():

        mae = mean_absolute_error(
            y_test,
            pred
        )

        rmse = np.sqrt(
            mean_squared_error(
                y_test,
                pred
            )
        )

        r2 = r2_score(
            y_test,
            pred
        )

        bias = np.mean(
            y_test - pred
        )


        results.append({
            "Training Window": window_name,
            "Model": model_name,
            "MAE": mae,
            "RMSE": rmse,
            "R2": r2,
            "Bias": bias
        })


# ============================================================
# 9. RESULTS TABLE
# ============================================================

test_results_df = (
    pd.DataFrame(results)
    .sort_values(
        ["MAE", "RMSE"],
        ascending=[True, True]
    )
    .reset_index(drop=True)
)


print("\n" + "=" * 80)
print("FINAL TEST SET MODEL COMPARISON")
print("=" * 80)

print(
    test_results_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.3f}"
    )
)

print("=" * 80)


# ============================================================
# 10. BEST TEST RESULT
# ============================================================

best_result = test_results_df.iloc[0]


print("\nBEST TEST RESULT")
print("-" * 40)

print(
    f"Training Window: "
    f"{best_result['Training Window']}"
)

print(
    f"Model:           "
    f"{best_result['Model']}"
)

print(
    f"MAE:             "
    f"{best_result['MAE']:.3f}"
)

print(
    f"RMSE:            "
    f"{best_result['RMSE']:.3f}"
)

print(
    f"R²:              "
    f"{best_result['R2']:.3f}"
)

print(
    f"Bias:            "
    f"{best_result['Bias']:+.3f}"
)


FINAL TESTING DATASET

10_years
Training dates: 2015-10-27 → 2025-04-16
Training observations: 253,009

15_years
Training dates: 2010-10-26 → 2025-04-16
Training observations: 374,979

all_history
Training dates: 1996-11-01 → 2025-04-16
Training observations: 703,253

TEST SET
Test dates: 2025-04-18 → 2026-04-17
Test observations: 26,833


######################################################################
FINAL TRAINING WINDOW: 10_YEARS
######################################################################

STARTING: HistGradientBoostingRegressor
TRAINING WINDOW: 10_years
Time: 09:27:20
----------------------------------------------------------------------
COMPLETED: HistGradientBoostingRegressor
TRAINING WINDOW: 10_years
Training time: 24.91 seconds
Time: 09:27:44


STARTING: XGBRegressor
TRAINING WINDOW: 10_years
Time: 09:27:44
----------------------------------------------------------------------
COMPLETED: XGBRegressor
TRAINING WINDOW: 10_years
Training time: 28.81 seconds
Tim

### 2.8 Optional tuning dependency
Install Optuna if required by the notebook environment.

In [ ]:
pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 442.4/442.4 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.7/268.7 kB 15.7 MB/s eta 0:00:00


### 2.9 LightGBM hyperparameter optimisation
Tune the LightGBM point estimator and retain its fitted configuration. **Important:** Negative Binomial modelling requires a conditional **mean** prediction; a model trained with an L1/median objective should not be used as the NB mean without refitting.

In [ ]:

import time
import numpy as np
import pandas as pd
import optuna
import lightgbm as lgb

from datetime import datetime
from lightgbm import LGBMRegressor

from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)


# ============================================================
# 1. CONFIGURATION
# ============================================================

N_TRIALS = 30
N_SPLITS = 3
EARLY_STOPPING_ROUNDS = 50
RANDOM_STATE = 42

# Must be sorted chronologically.
# Do not include the final test period in this data.

X_tune = X_train_final.astype(np.float32)
y_tune = np.asarray(
    y_train_final,
    dtype=np.float32
)

tscv = TimeSeriesSplit(
    n_splits=N_SPLITS
)


# ============================================================
# 2. OPTUNA OBJECTIVE
# ============================================================

def objective(trial):

    params = {

        "objective": "regression_l1",

        "learning_rate": trial.suggest_float(
            "learning_rate",
            0.01,
            0.10,
            log=True
        ),

        "num_leaves": trial.suggest_int(
            "num_leaves",
            15,
            127
        ),

        "max_depth": trial.suggest_categorical(
            "max_depth",
            [-1, 4, 6, 8, 10]
        ),

        "min_child_samples": trial.suggest_int(
            "min_child_samples",
            20,
            100
        ),

        "reg_alpha": trial.suggest_float(
            "reg_alpha",
            0.0,
            5.0
        ),

        "reg_lambda": trial.suggest_float(
            "reg_lambda",
            0.0,
            10.0
        ),

        "colsample_bytree": trial.suggest_float(
            "colsample_bytree",
            0.6,
            1.0
        ),

        "subsample": trial.suggest_float(
            "subsample",
            0.6,
            1.0
        ),

        "subsample_freq": 1,

        "max_bin": 127,

        # Early stopping determines the effective tree count.
        "n_estimators": 1500,

        "random_state": RANDOM_STATE,
        "n_jobs": -1,
        "verbosity": -1
    }

    fold_maes = []
    fold_best_iterations = []

    for fold, (train_idx, valid_idx) in enumerate(
        tscv.split(X_tune),
        start=1
    ):

        X_fold_train = X_tune.iloc[train_idx]
        X_fold_valid = X_tune.iloc[valid_idx]

        y_fold_train = y_tune[train_idx]
        y_fold_valid = y_tune[valid_idx]

        model = LGBMRegressor(**params)

        model.fit(
            X_fold_train,
            y_fold_train,

            eval_set=[
                (X_fold_valid, y_fold_valid)
            ],

            eval_metric="l1",

            callbacks=[
                lgb.early_stopping(
                    stopping_rounds=EARLY_STOPPING_ROUNDS,
                    verbose=False
                ),
                lgb.log_evaluation(period=0)
            ]
        )

        pred = model.predict(
            X_fold_valid,
            num_iteration=model.best_iteration_
        )

        mae = mean_absolute_error(
            y_fold_valid,
            pred
        )

        fold_maes.append(mae)
        fold_best_iterations.append(
            model.best_iteration_
        )

        # Report cumulative CV performance to Optuna.
        trial.report(
            float(np.mean(fold_maes)),
            step=fold
        )

        # Stop evaluating clearly unpromising trials.
        if trial.should_prune():
            raise optuna.TrialPruned()

    trial.set_user_attr(
        "best_iterations",
        fold_best_iterations
    )

    trial.set_user_attr(
        "mean_best_iteration",
        int(np.median(fold_best_iterations))
    )

    return float(np.mean(fold_maes))


# ============================================================
# 3. CREATE OPTUNA STUDY
# ============================================================

sampler = optuna.samplers.TPESampler(
    seed=RANDOM_STATE
)

pruner = optuna.pruners.MedianPruner(
    n_startup_trials=10,
    n_warmup_steps=1
)

study = optuna.create_study(
    direction="minimize",
    sampler=sampler,
    pruner=pruner,
    study_name="NBA_Points_LGBM"
)


# ============================================================
# 4. RUN HYPERPARAMETER TUNING
# ============================================================

print("\n" + "=" * 70)
print("STARTING OPTUNA LIGHTGBM HYPERPARAMETER TUNING")
print(f"Time: {datetime.now():%H:%M:%S}")
print(f"Trials: {N_TRIALS}")
print(f"CV Folds: {N_SPLITS}")
print("=" * 70)

start_time = time.time()

study.optimize(
    objective,
    n_trials=N_TRIALS,
    n_jobs=1,
    show_progress_bar=True
)

elapsed = time.time() - start_time


print("\n" + "=" * 70)
print("OPTUNA TUNING COMPLETE")
print(f"Total tuning time: {elapsed / 60:.2f} minutes")
print("=" * 70)


# ============================================================
# 5. BEST PARAMETERS
# ============================================================

print("\nBEST PARAMETERS")
print("-" * 70)

for parameter, value in study.best_params.items():
    print(f"{parameter}: {value}")

print(
    f"\nBest CV MAE: "
    f"{study.best_value:.3f}"
)


# ============================================================
# 6. DETERMINE OPTIMAL BOOSTING ITERATIONS
# ============================================================

best_trial = study.best_trial

best_iterations = best_trial.user_attrs[
    "best_iterations"
]

optimal_iterations = max(
    1,
    int(np.median(best_iterations))
)

print("\nBEST ITERATIONS BY FOLD")
print(best_iterations)

print(
    f"Selected final iterations: "
    f"{optimal_iterations}"
)


# ============================================================
# 7. RETRAIN BEST MODEL
# ============================================================

best_params = study.best_params.copy()

best_params.update({

    "objective": "regression_l1",

    "n_estimators": optimal_iterations,

    "subsample_freq": 1,

    "max_bin": 127,

    "random_state": RANDOM_STATE,

    "n_jobs": -1,

    "verbosity": -1
})


best_lgbm_model = LGBMRegressor(
    **best_params
)

print("\n" + "=" * 70)
print("RETRAINING BEST LIGHTGBM MODEL")
print("=" * 70)

start_time = time.time()

best_lgbm_model.fit(
    X_tune,
    y_tune
)

print(
    f"Final training time: "
    f"{time.time() - start_time:.2f} seconds"
)


# ============================================================
# 8. OPTIONAL VALIDATION EVALUATION
# ============================================================

# IMPORTANT:
# Only use X_validation if those observations were NOT
# included in X_train_final / y_train_final.
#
# Otherwise, evaluate using the CV results above.


# ============================================================
# 9. SAVE SEARCH RESULTS
# ============================================================

optuna_results_df = study.trials_dataframe()

print("\nTOP 10 TRIALS")
print("-" * 70)

print(
    optuna_results_df[
        optuna_results_df["state"] == "COMPLETE"
    ]
    .sort_values("value")
    .head(10)[
        ["number", "value", "duration", "state"]
    ]
    .to_string(index=False)
)


# ============================================================
# 10. FINAL MODEL READY
# ============================================================

print("\n" + "=" * 70)
print("BEST LIGHTGBM MODEL READY")
print("=" * 70)

print(f"Best CV MAE: {study.best_value:.3f}")
print(f"Number of trees: {optimal_iterations}")
print(f"Completed at: {datetime.now():%H:%M:%S}")
print("=" * 70)


[I 2026-10-12 09:34:32,196] A new study created in memory with name: NBA_Points_LGBM



STARTING OPTUNA LIGHTGBM HYPERPARAMETER TUNING
Time: 09:34:32
Trials: 30
CV Folds: 3


  0%|          | 0/30 [00:00<?, ?it/s]

[I 2026-10-12 09:38:18,347] Trial 0 finished with value: 4.373514856991623 and parameters: {'learning_rate': 0.023688639503640783, 'num_leaves': 122, 'max_depth': -1, 'min_child_samples': 90, 'reg_alpha': 3.005575058716044, 'reg_lambda': 7.080725777960454, 'colsample_bytree': 0.608233797718321, 'subsample': 0.9879639408647978}. Best is trial 0 with value: 4.373514856991623.
[I 2026-10-12 09:39:14,793] Trial 1 finished with value: 4.373212348167115 and parameters: {'learning_rate': 0.06798962421591129, 'num_leaves': 38, 'max_depth': 8, 'min_child_samples': 43, 'reg_alpha': 3.0592644736118975, 'reg_lambda': 1.3949386065204183, 'colsample_bytree': 0.7168578594140873, 'subsample': 0.7465447373174767}. Best is trial 1 with value: 4.373212348167115.
[I 2026-10-12 09:42:22,254] Trial 2 finished with value: 4.3745126578036375 and parameters: {'learning_rate': 0.028580510658069373, 'num_leaves': 103, 'max_depth': 10, 'min_child_samples': 33, 'reg_alpha': 0.3252579649263976, 'reg_lambda': 9.4888

---
## 3. Probabilistic points modelling
Translate expected points into an entire count distribution and over/under probabilities.

### 3.1 Out-of-fold conditional mean predictions
Generate chronological OOF expected-points predictions to fit dispersion without using in-sample mean predictions. The training observations and feature rows must remain aligned.

In [ ]:

import numpy as np
import pandas as pd
import lightgbm as lgb

from sklearn.base import clone
from sklearn.model_selection import TimeSeriesSplit

# ============================================================
# 1. PREPARE DATA
# ============================================================

X_nb_train = X_train_final.astype(np.float32)
y_nb_train = np.asarray(y_train_final, dtype=np.float64)

# LightGBM must predict the conditional mean, not median.
mean_model_template = clone(best_lgbm_model)

mean_model_template.set_params(
    objective="regression",
    random_state=42,
    verbosity=-1
)

# ============================================================
# 2. CHRONOLOGICAL OOF PREDICTIONS
# ============================================================

tscv = TimeSeriesSplit(n_splits=5)

oof_mu = np.full(
    len(y_nb_train),
    np.nan,
    dtype=np.float64
)

for fold, (train_idx, valid_idx) in enumerate(
    tscv.split(X_nb_train),
    start=1
):
    print(f"Starting OOF fold {fold}/5")

    model = clone(mean_model_template)

    model.fit(
        X_nb_train.iloc[train_idx],
        y_nb_train[train_idx]
    )

    pred = model.predict(
        X_nb_train.iloc[valid_idx]
    )

    oof_mu[valid_idx] = np.maximum(pred, 1e-6)

    print(
        f"Completed fold {fold} | "
        f"Observations: {len(valid_idx):,}"
    )

# ============================================================
# 3. EXTRACT VALID OOF OBSERVATIONS
# ============================================================

valid_mask = (
    np.isfinite(oof_mu)
    & np.isfinite(y_nb_train)
    & (y_nb_train >= 0)
)

y_oof = y_nb_train[valid_mask]
mu_oof = oof_mu[valid_mask]

print("\nOOF DATA READY")
print(f"Observations: {len(y_oof):,}")
print(f"Mean actual points: {y_oof.mean():.3f}")
print(f"Mean predicted points: {mu_oof.mean():.3f}")


Starting OOF fold 1/5
Completed fold 1 | Observations: 117,208
Starting OOF fold 2/5
Completed fold 2 | Observations: 117,208
Starting OOF fold 3/5
Completed fold 3 | Observations: 117,208
Starting OOF fold 4/5
Completed fold 4 | Observations: 117,208
Starting OOF fold 5/5
Completed fold 5 | Observations: 117,208

OOF DATA READY
Observations: 586,040
Mean actual points: 10.035
Mean predicted points: 10.039


### 3.2 Global Negative Binomial baseline
Fit one dispersion parameter (α) from OOF outcomes, train the final mean model, and calculate Negative Binomial probabilities. Under the NB2 parameterisation, **variance = μ + αμ²**.

In [ ]:

from scipy.optimize import minimize_scalar
from scipy.stats import nbinom

# ============================================================
# 4. NEGATIVE BINOMIAL NEGATIVE LOG-LIKELIHOOD
# ============================================================

def nb_negative_log_likelihood(log_alpha, y, mu):

    alpha = np.exp(log_alpha)

    r = 1.0 / alpha

    p = r / (r + mu)

    log_likelihood = nbinom.logpmf(
        y,
        n=r,
        p=p
    )

    return -np.sum(log_likelihood)


# ============================================================
# 5. ESTIMATE DISPERSION
# ============================================================

result = minimize_scalar(
    nb_negative_log_likelihood,
    bounds=(np.log(1e-5), np.log(10.0)),
    args=(y_oof, mu_oof),
    method="bounded",
    options={"xatol": 1e-6}
)

if not result.success:
    raise RuntimeError(result.message)

alpha_hat = float(np.exp(result.x))

print("\n" + "=" * 60)
print("NEGATIVE BINOMIAL DISPERSION FIT")
print("=" * 60)

print(f"Estimated alpha: {alpha_hat:.6f}")
print(f"Estimated r:     {1 / alpha_hat:.4f}")

print("=" * 60)


# ============================================================
# 6. FINAL LIGHTGBM MEAN MODEL
# ============================================================

final_mean_model = clone(mean_model_template)

final_mean_model.fit(
    X_nb_train,
    y_nb_train
)

# Predict expected points for test observations.
X_nb_test = X_test.astype(np.float32)

mu_test = np.maximum(
    final_mean_model.predict(X_nb_test),
    1e-6
)

# Convert NB parameters.
r_test = 1.0 / alpha_hat

p_test = r_test / (r_test + mu_test)

print("Final mean model fitted.")
print(f"Test predictions generated: {len(mu_test):,}")



from scipy.stats import nbinom

# ============================================================
# 7. PROBABILITY FUNCTIONS
# ============================================================

def nb_over_probability(mu, alpha, line):

    mu = np.maximum(np.asarray(mu), 1e-6)

    r = 1.0 / alpha
    p = r / (r + mu)

    # Over L means Y > L.
    # Works for integer and half-point lines.
    return nbinom.sf(
        np.floor(line),
        n=r,
        p=p
    )


def nb_under_probability(mu, alpha, line):

    mu = np.maximum(np.asarray(mu), 1e-6)

    r = 1.0 / alpha
    p = r / (r + mu)

    # Under L means Y < L.
    return nbinom.cdf(
        np.ceil(line) - 1,
        n=r,
        p=p
    )


def nb_push_probability(mu, alpha, line):

    mu = np.maximum(np.asarray(mu), 1e-6)

    r = 1.0 / alpha
    p = r / (r + mu)

    # Push possible only on integer lines.
    if float(line).is_integer():
        return nbinom.pmf(
            int(line),
            n=r,
            p=p
        )

    return np.zeros_like(mu, dtype=float)


# ============================================================
# 8. EXAMPLE: PLAYER EXPECTED TO SCORE 24 POINTS
# ============================================================

expected_points = 24.0
points_line = 22.5

prob_over = nb_over_probability(
    expected_points,
    alpha_hat,
    points_line
)

prob_under = nb_under_probability(
    expected_points,
    alpha_hat,
    points_line
)

print(f"Expected points: {expected_points:.2f}")
print(f"Line:            {points_line:.1f}")
print(f"P(Over):         {float(prob_over):.2%}")
print(f"P(Under):        {float(prob_under):.2%}")



# ============================================================
# 9. TEST-SET PROBABILITIES
# ============================================================

# Example: a common line for all observations.
# In production, replace this with each player's actual line.

points_line = 19.5

prob_over_test = nb_over_probability(
    mu_test,
    alpha_hat,
    points_line
)

prob_under_test = nb_under_probability(
    mu_test,
    alpha_hat,
    points_line
)

nb_predictions_df = pd.DataFrame({

    "actual_points": y_test,

    "predicted_points": mu_test,

    "predicted_variance": (
        mu_test + alpha_hat * mu_test**2
    ),

    "predicted_sd": np.sqrt(
        mu_test + alpha_hat * mu_test**2
    ),

    "points_line": points_line,

    "prob_over": prob_over_test,

    "prob_under": prob_under_test
})

print(
    nb_predictions_df.head(20).to_string(
        index=False,
        float_format=lambda x: f"{x:.3f}"
    )
)



from scipy.stats import nbinom
from sklearn.metrics import brier_score_loss

# ============================================================
# 10. NEGATIVE BINOMIAL LOG-LIKELIHOOD
# ============================================================

nb_log_likelihood = nbinom.logpmf(
    y_test,
    n=r_test,
    p=p_test
)

nb_nll = -np.mean(nb_log_likelihood)

print(f"Test Negative Log-Likelihood: {nb_nll:.4f}")


# ============================================================
# 11. OVER/UNDER BRIER SCORE
# ============================================================

# Only evaluate lines where an over/under has
# a binary outcome (no pushes).

line = 19.5

actual_over = (
    np.asarray(y_test) > line
).astype(int)

predicted_over = nb_over_probability(
    mu_test,
    alpha_hat,
    line
)

brier = brier_score_loss(
    actual_over,
    predicted_over
)

print(f"Brier Score (Over {line}): {brier:.4f}")


# ============================================================
# 12. CALIBRATION TABLE
# ============================================================

calibration_df = pd.DataFrame({
    "predicted_probability": predicted_over,
    "actual_over": actual_over
})

calibration_df["probability_bucket"] = pd.cut(
    calibration_df["predicted_probability"],
    bins=np.linspace(0, 1, 11),
    include_lowest=True
)

calibration_summary = (
    calibration_df
    .groupby(
        "probability_bucket",
        observed=True
    )
    .agg(
        n=("actual_over", "size"),
        predicted_probability=("predicted_probability", "mean"),
        actual_frequency=("actual_over", "mean")
    )
    .reset_index()
)

print("\nPROBABILITY CALIBRATION")
print("-" * 70)

print(
    calibration_summary.to_string(
        index=False,
        float_format=lambda x: f"{x:.3f}"
    )
)




NEGATIVE BINOMIAL DISPERSION FIT
Estimated alpha: 0.331853
Estimated r:     3.0134
Final mean model fitted.
Test predictions generated: 26,833
Expected points: 24.00
Line:            22.5
P(Over):         46.40%
P(Under):        53.60%
 actual_points  predicted_points  predicted_variance  predicted_sd  points_line  prob_over  prob_under
         5.000             9.000              35.881         5.990       19.500      0.060       0.940
         5.000             8.352              31.501         5.613       19.500      0.045       0.955
        20.000            19.965             152.242        12.339       19.500      0.438       0.562
        15.000            13.524              74.221         8.615       19.500      0.210       0.790
        20.000            10.420              46.455         6.816       19.500      0.101       0.899
        17.000            22.217             186.020        13.639       19.500      0.505       0.495
        29.000            25.770          

### 3.3 Probability evaluation at multiple point lines
Compare binary log-loss, Brier score, observed over rates and predicted over rates. A baseline calculated using test outcomes is **diagnostic only**, not a deployable forecasting benchmark.

In [ ]:

import numpy as np
import pandas as pd

from sklearn.metrics import (
    log_loss,
    brier_score_loss
)


# ============================================================
# 1. CONFIGURATION
# ============================================================

# Initial evaluation line.
# Later replace with actual historical Sportsbet lines.

POINTS_LINE = 22.5


# ============================================================
# 2. NEGATIVE BINOMIAL PROBABILITIES
# ============================================================

predicted_over = nb_over_probability(
    mu=mu_test,
    alpha=alpha_hat,
    line=POINTS_LINE
)

predicted_over = np.asarray(
    predicted_over,
    dtype=float
)


# ============================================================
# 3. ACTUAL BETTING OUTCOMES
# ============================================================

actual_points = np.asarray(y_test)

actual_over = (
    actual_points > POINTS_LINE
).astype(int)


# ============================================================
# 4. LOG-LOSS
# ============================================================

# Avoid numerical problems with log(0).

eps = 1e-15

predicted_over_safe = np.clip(
    predicted_over,
    eps,
    1 - eps
)

nb_log_loss = log_loss(
    actual_over,
    predicted_over_safe,
    labels=[0, 1]
)


# ============================================================
# 5. BRIER SCORE
# ============================================================

nb_brier = brier_score_loss(
    actual_over,
    predicted_over
)


# ============================================================
# 6. BASELINE LOG-LOSS
# ============================================================

# Constant probability equal to the observed over rate.
# Diagnostic only: this uses test outcomes to estimate
# the baseline and is not a deployable pregame benchmark.

historical_over_rate = actual_over.mean()

baseline_probs = np.full(
    len(actual_over),
    historical_over_rate
)

baseline_log_loss = log_loss(
    actual_over,
    baseline_probs,
    labels=[0, 1]
)


# ============================================================
# 7. PRINT RESULTS
# ============================================================

print("\n" + "=" * 70)
print("NEGATIVE BINOMIAL BETTING PROBABILITY EVALUATION")
print("=" * 70)

print(f"Points Line:          {POINTS_LINE}")
print(f"Observations:         {len(actual_over):,}")

print("\nMODEL PERFORMANCE")
print("-" * 70)

print(f"Log-Loss:             {nb_log_loss:.5f}")
print(f"Brier Score:          {nb_brier:.5f}")

print("\nCONSTANT-RATE BASELINE")
print("-" * 70)

print(f"Observed Over Rate:   {historical_over_rate:.2%}")
print(f"Baseline Log-Loss:    {baseline_log_loss:.5f}")

print("\nIMPROVEMENT")
print("-" * 70)

print(
    f"Log-Loss Improvement: "
    f"{baseline_log_loss - nb_log_loss:+.5f}"
)

print("=" * 70)

# ============================================================
# MULTIPLE POINTS LINE EVALUATION
# ============================================================

POINTS_LINES = [
    4.5,
    9.5,
    14.5,
    19.5,
    24.5,
    29.5,
    34.5
]

results = []

for line in POINTS_LINES:

    # Model probability
    prob_over = nb_over_probability(
        mu_test,
        alpha_hat,
        line
    )

    prob_over = np.asarray(
        prob_over,
        dtype=float
    )

    # Actual outcome
    actual_over = (
        np.asarray(y_test) > line
    ).astype(int)

    # Log-loss
    ll = log_loss(
        actual_over,
        np.clip(prob_over, 1e-15, 1 - 1e-15),
        labels=[0, 1]
    )

    # Brier score
    brier = brier_score_loss(
        actual_over,
        prob_over
    )

    # Probability calibration bias
    probability_bias = (
        prob_over.mean() - actual_over.mean()
    )

    results.append({
        "Points Line": line,
        "Observations": len(actual_over),
        "Actual Over %": actual_over.mean() * 100,
        "Predicted Over %": prob_over.mean() * 100,
        "Log-Loss": ll,
        "Brier": brier,
        "Probability Bias": probability_bias
    })


logloss_results_df = (
    pd.DataFrame(results)
    .sort_values("Points Line")
)

print("\n" + "=" * 90)
print("NEGATIVE BINOMIAL LOG-LOSS BY POINTS LINE")
print("=" * 90)

print(
    logloss_results_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)

print("=" * 90)



NEGATIVE BINOMIAL BETTING PROBABILITY EVALUATION
Points Line:          22.5
Observations:         26,833

MODEL PERFORMANCE
----------------------------------------------------------------------
Log-Loss:             0.21944
Brier Score:          0.06522

CONSTANT-RATE BASELINE
----------------------------------------------------------------------
Observed Over Rate:   10.29%
Baseline Log-Loss:    0.33148

IMPROVEMENT
----------------------------------------------------------------------
Log-Loss Improvement: +0.11204

NEGATIVE BINOMIAL LOG-LOSS BY POINTS LINE
 Points Line  Observations  Actual Over %  Predicted Over %  Log-Loss  Brier  Probability Bias
      4.5000         26833        71.4605           70.2866    0.4372 0.1419           -0.0117
      9.5000         26833        48.0677           41.8609    0.5001 0.1632           -0.0621
     14.5000         26833        28.8004           24.7801    0.4184 0.1331           -0.0402
     19.5000         26833        15.7045           

### 3.4 Poisson versus Negative Binomial
Compare the equidispersed Poisson benchmark with the overdispersed global NB model across scoring thresholds.

In [ ]:

import numpy as np
import pandas as pd

from scipy.stats import poisson, nbinom
from sklearn.metrics import log_loss, brier_score_loss


# ============================================================
# COMPARE POISSON VS NEGATIVE BINOMIAL
# ============================================================

POINTS_LINES = [
    4.5, 9.5, 14.5, 19.5,
    24.5, 29.5, 34.5
]

mu = np.maximum(
    np.asarray(mu_test, dtype=float),
    1e-6
)

actual_points = np.asarray(y_test)

results = []

for line in POINTS_LINES:

    actual_over = (
        actual_points > line
    ).astype(int)

    # Poisson probabilities
    poisson_prob = poisson.sf(
        np.floor(line),
        mu=mu
    )

    # Negative Binomial probabilities
    nb_prob = nb_over_probability(
        mu,
        alpha_hat,
        line
    )

    for name, prob in [
        ("Poisson", poisson_prob),
        ("Negative Binomial", nb_prob)
    ]:

        results.append({
            "Distribution": name,
            "Line": line,
            "Log-Loss": log_loss(
                actual_over,
                np.clip(prob, 1e-15, 1 - 1e-15),
                labels=[0, 1]
            ),
            "Brier": brier_score_loss(
                actual_over,
                prob
            ),
            "Actual Over %": actual_over.mean() * 100,
            "Predicted Over %": prob.mean() * 100,
            "Bias (pp)": (
                prob.mean() - actual_over.mean()
            ) * 100
        })


comparison_df = pd.DataFrame(results)

print(
    comparison_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)


     Distribution    Line  Log-Loss  Brier  Actual Over %  Predicted Over %  Bias (pp)
          Poisson  4.5000    0.5435 0.1556        71.4605           80.5943     9.1338
Negative Binomial  4.5000    0.4372 0.1419        71.4605           70.2866    -1.1739
          Poisson  9.5000    0.5657 0.1675        48.0677           48.4527     0.3850
Negative Binomial  9.5000    0.5001 0.1632        48.0677           41.8609    -6.2067
          Poisson 14.5000    0.5064 0.1370        28.8004           25.4994    -3.3010
Negative Binomial 14.5000    0.4184 0.1331        28.8004           24.7801    -4.0203
          Poisson 19.5000    0.3757 0.0920        15.7045           12.4502    -3.2543
Negative Binomial 19.5000    0.2918 0.0893        15.7045           14.8538    -0.8508
          Poisson 24.5000    0.2391 0.0533         7.7740            5.3352    -2.4388
Negative Binomial 24.5000    0.1823 0.0529         7.7740            9.0322     1.2582
          Poisson 29.5000    0.1391 0.0283 

### 3.5 Bucket-conditioned dispersion
Estimate α separately for ranges of predicted expected points using OOF data. This is the simpler conditional NB benchmark.

In [ ]:

import numpy as np
import pandas as pd

from scipy.optimize import minimize_scalar
from scipy.stats import nbinom


# ============================================================
# 1. DISPERSION ESTIMATION FUNCTION
# ============================================================

def estimate_nb_alpha(y, mu):

    y = np.asarray(y, dtype=float)
    mu = np.maximum(np.asarray(mu, dtype=float), 1e-6)

    def objective(log_alpha):

        alpha = np.exp(log_alpha)

        r = 1.0 / alpha
        p = r / (r + mu)

        return -np.mean(
            nbinom.logpmf(
                y,
                n=r,
                p=p
            )
        )

    result = minimize_scalar(
        objective,
        bounds=(np.log(1e-5), np.log(10)),
        method="bounded"
    )

    if not result.success:
        raise RuntimeError(result.message)

    return float(np.exp(result.x))


# ============================================================
# 2. PREPARE OOF DATA
# ============================================================

dispersion_df = pd.DataFrame({

    "actual_points": y_oof,

    "predicted_points": mu_oof

})

# Fixed buckets based on predicted expected points.
POINTS_BINS = [
    0, 5, 10, 15, 20, 25, 30, np.inf
]

dispersion_df["points_bucket"] = pd.cut(
    dispersion_df["predicted_points"],
    bins=POINTS_BINS,
    include_lowest=True
)


# ============================================================
# 3. FIT ALPHA FOR EACH BUCKET
# ============================================================

MIN_BUCKET_SIZE = 200

bucket_alphas = {}

for bucket, group in dispersion_df.groupby(
    "points_bucket",
    observed=True
):

    if len(group) < MIN_BUCKET_SIZE:
        alpha = alpha_hat
    else:
        alpha = estimate_nb_alpha(
            group["actual_points"].to_numpy(),
            group["predicted_points"].to_numpy()
        )

    bucket_alphas[bucket] = alpha

    print(
        f"Bucket: {str(bucket):>15} | "
        f"Observations: {len(group):>7,} | "
        f"Alpha: {alpha:.5f}"
    )


# ============================================================
# 4. ASSIGN TEST-SET DISPERSION
# ============================================================

test_buckets = pd.cut(
    mu_test,
    bins=POINTS_BINS,
    include_lowest=True
)

alpha_test = np.array([
    bucket_alphas[bucket]
    for bucket in test_buckets
])


# ============================================================
# 5. PREDICT PROBABILITIES
# ============================================================

def nb_over_conditional(mu, alpha, line):

    mu = np.maximum(
        np.asarray(mu, dtype=float),
        1e-6
    )

    alpha = np.maximum(
        np.asarray(alpha, dtype=float),
        1e-6
    )

    r = 1.0 / alpha
    p = r / (r + mu)

    return nbinom.sf(
        np.floor(line),
        n=r,
        p=p
    )


Bucket:   (-0.001, 5.0] | Observations: 128,371 | Alpha: 1.17494
Bucket:     (5.0, 10.0] | Observations: 210,995 | Alpha: 0.45093
Bucket:    (10.0, 15.0] | Observations: 133,255 | Alpha: 0.21125
Bucket:    (15.0, 20.0] | Observations:  68,453 | Alpha: 0.12773
Bucket:    (20.0, 25.0] | Observations:  32,578 | Alpha: 0.09081
Bucket:    (25.0, 30.0] | Observations:  11,169 | Alpha: 0.07260
Bucket:     (30.0, inf] | Observations:   1,219 | Alpha: 0.07309


### 3.6 Global versus bucket-conditioned NB
Compare the two NB formulations on identical test observations and thresholds.

In [ ]:

from sklearn.metrics import log_loss, brier_score_loss

# ============================================================
# GLOBAL VS CONDITIONAL NB COMPARISON
# ============================================================

results = []

for line in [
    4.5, 9.5, 14.5, 19.5,
    24.5, 29.5, 34.5
]:

    actual_over = (
        np.asarray(y_test) > line
    ).astype(int)

    global_prob = nb_over_probability(
        mu_test,
        alpha_hat,
        line
    )

    conditional_prob = nb_over_conditional(
        mu_test,
        alpha_test,
        line
    )

    for model_name, prob in [
        ("Global NB", global_prob),
        ("Conditional NB", conditional_prob)
    ]:

        results.append({

            "Model": model_name,

            "Line": line,

            "Log-Loss": log_loss(
                actual_over,
                np.clip(prob, 1e-15, 1 - 1e-15),
                labels=[0, 1]
            ),

            "Brier": brier_score_loss(
                actual_over,
                prob
            ),

            "Actual Over %": actual_over.mean() * 100,

            "Predicted Over %": np.mean(prob) * 100

        })


comparison_df = pd.DataFrame(results)

print(
    comparison_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)


         Model    Line  Log-Loss  Brier  Actual Over %  Predicted Over %
     Global NB  4.5000    0.4372 0.1419        71.4605           70.2866
Conditional NB  4.5000    0.4350 0.1417        71.4605           70.8843
     Global NB  9.5000    0.5001 0.1632        48.0677           41.8609
Conditional NB  9.5000    0.4842 0.1593        48.0677           46.4090
     Global NB 14.5000    0.4184 0.1331        28.8004           24.7801
Conditional NB 14.5000    0.4061 0.1295        28.8004           28.2648
     Global NB 19.5000    0.2918 0.0893        15.7045           14.8538
Conditional NB 19.5000    0.2859 0.0872        15.7045           16.0427
     Global NB 24.5000    0.1823 0.0529         7.7740            9.0322
Conditional NB 24.5000    0.1777 0.0515         7.7740            8.4621
     Global NB 29.5000    0.1077 0.0286         3.5516            5.5615
Conditional NB 29.5000    0.1007 0.0273         3.5516            4.1407
     Global NB 34.5000    0.0584 0.0135         1.3

---
## 4. V1 conditional dispersion model
Fit a second LightGBM model that predicts the NB shape parameter conditional on player-game features and expected points.

**Method note:** This experiment uses a custom NB likelihood objective and a positive-clipped Hessian. Initial-score handling and derivatives should be verified before treating the exported model as production-ready.

In [ ]:

import numpy as np
import pandas as pd
import lightgbm as lgb

from scipy.special import digamma, polygamma, gammaln
from scipy.stats import nbinom
from sklearn.metrics import log_loss, brier_score_loss


# ============================================================
# 1. PREPARE DISPERSION TRAINING DATA
# ============================================================

# Assumes:
# - X_nb_train is aligned with the original training rows
# - oof_mu contains chronological OOF mean predictions
# - y_nb_train contains actual points
# - final_mean_model is already fitted
# - X_nb_test contains test features
# - alpha_hat is your previously fitted global NB dispersion

valid_mask = (
    np.isfinite(oof_mu)
    & np.isfinite(y_nb_train)
    & (oof_mu > 0)
    & (y_nb_train >= 0)
)

X_dispersion = (
    X_nb_train.loc[valid_mask]
    .copy()
    .reset_index(drop=True)
)

mu_dispersion = np.asarray(
    oof_mu[valid_mask],
    dtype=np.float64
)

y_dispersion = np.asarray(
    y_nb_train[valid_mask],
    dtype=np.float64
)

# Add the OOF expected points as a dispersion feature.
X_dispersion["predicted_points"] = mu_dispersion

# NB shape parameter r = 1 / alpha.
# LightGBM predicts log(r), ensuring r > 0.

INITIAL_LOG_R = np.log(1.0 / alpha_hat)


# ============================================================
# 2. CUSTOM NEGATIVE BINOMIAL OBJECTIVE
# ============================================================

def make_nb_dispersion_objective(y_actual, mu_fixed):

    y_actual = np.asarray(y_actual, dtype=np.float64)
    mu_fixed = np.maximum(
        np.asarray(mu_fixed, dtype=np.float64),
        1e-6
    )

    def objective(pred_log_r, dataset):

        # Numerically constrain the shape parameter.
        log_r = np.clip(pred_log_r, -8.0, 12.0)
        r = np.exp(log_r)

        # First derivative of NLL with respect to r.
        grad_r = (
            -digamma(y_actual + r)
            + digamma(r)
            - np.log(r)
            + np.log(r + mu_fixed)
            + (r + y_actual) / (r + mu_fixed)
            - 1.0
        )

        # Second derivative with respect to r.
        hess_r = (
            -polygamma(1, y_actual + r)
            + polygamma(1, r)
            - 1.0 / r
            + 1.0 / (r + mu_fixed)
            + (mu_fixed - y_actual) / (r + mu_fixed)**2
        )

        # Chain rule for log(r).
        grad = grad_r * r
        hess = hess_r * r**2 + grad_r * r

        # LightGBM requires positive Hessians for stable splits.
        # This is a clipped curvature approximation.
        hess = np.maximum(hess, 1e-4)

        return grad, hess

    return objective


# ============================================================
# 3. TRAIN DISPERSION MODEL
# ============================================================

# Use the native LightGBM API so we can initialise
# the predictions at the global dispersion estimate.

dispersion_dataset = lgb.Dataset(
    X_dispersion,
    label=y_dispersion,
    free_raw_data=False
)

dispersion_dataset.set_init_score(
    np.full(
        len(y_dispersion),
        INITIAL_LOG_R
    )
)

dispersion_params = {
    "objective": make_nb_dispersion_objective(
        y_dispersion,
        mu_dispersion
    ),
    "learning_rate": 0.02,
    "num_leaves": 15,
    "max_depth": 4,
    "min_data_in_leaf": 200,
    "lambda_l2": 5.0,
    "feature_fraction": 0.8,
    "verbosity": -1,
    "num_threads": -1,
    "seed": 42
}

dispersion_model = lgb.train(
    params=dispersion_params,
    train_set=dispersion_dataset,
    num_boost_round=150
)


# ============================================================
# 4. PREDICT TEST-SET DISPERSION
# ============================================================

mu_test = np.maximum(
    final_mean_model.predict(X_nb_test),
    1e-6
)

X_dispersion_test = X_nb_test.copy()
X_dispersion_test["predicted_points"] = mu_test

# Raw predictions exclude init_score.
# Add the initial log(r) offset explicitly.
predicted_log_r = (
    INITIAL_LOG_R
    + dispersion_model.predict(
        X_dispersion_test,
        raw_score=True
    )
)

predicted_log_r = np.clip(
    predicted_log_r,
    -8.0,
    12.0
)

r_test_conditional = np.exp(predicted_log_r)

alpha_test_conditional = 1.0 / r_test_conditional

variance_test_conditional = (
    mu_test
    + alpha_test_conditional * mu_test**2
)


# ============================================================
# 5. NEGATIVE BINOMIAL PROBABILITIES
# ============================================================

def nb_over_dispersion(mu, alpha, line):

    mu = np.maximum(
        np.asarray(mu, dtype=float),
        1e-6
    )

    alpha = np.maximum(
        np.asarray(alpha, dtype=float),
        1e-6
    )

    r = 1.0 / alpha
    p = r / (r + mu)

    return nbinom.sf(
        np.floor(line),
        n=r,
        p=p
    )


# ============================================================
# 6. EVALUATE LOG-LOSS AND BRIER SCORE
# ============================================================

POINTS_LINES = [
    4.5, 9.5, 14.5, 19.5,
    24.5, 29.5, 34.5
]

results = []

for line in POINTS_LINES:

    actual_over = (
        np.asarray(y_test) > line
    ).astype(int)

    global_prob = nb_over_dispersion(
        mu_test,
        alpha_hat,
        line
    )

    conditional_prob = nb_over_dispersion(
        mu_test,
        alpha_test_conditional,
        line
    )

    for model_name, prob in [
        ("Global NB", global_prob),
        ("LightGBM Conditional NB", conditional_prob)
    ]:

        results.append({
            "Model": model_name,
            "Line": line,
            "Log-Loss": log_loss(
                actual_over,
                np.clip(prob, 1e-15, 1 - 1e-15),
                labels=[0, 1]
            ),
            "Brier": brier_score_loss(
                actual_over,
                prob
            ),
            "Actual Over %": actual_over.mean() * 100,
            "Predicted Over %": np.mean(prob) * 100
        })


comparison_df = pd.DataFrame(results)

print("\n" + "=" * 90)
print("LIGHTGBM CONDITIONAL DISPERSION RESULTS")
print("=" * 90)

print(
    comparison_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)


# ============================================================
# 7. FULL DISTRIBUTION NEGATIVE LOG-LIKELIHOOD
# ============================================================

p_conditional = (
    r_test_conditional
    / (r_test_conditional + mu_test)
)

conditional_nb_nll = -np.mean(
    nbinom.logpmf(
        y_test,
        n=r_test_conditional,
        p=p_conditional
    )
)

print(
    f"\nConditional NB NLL: "
    f"{conditional_nb_nll:.5f}"
)

print(
    f"Mean predicted alpha: "
    f"{alpha_test_conditional.mean():.5f}"
)

print(
    f"Median predicted alpha: "
    f"{np.median(alpha_test_conditional):.5f}"
)



LIGHTGBM CONDITIONAL DISPERSION RESULTS
                  Model    Line  Log-Loss  Brier  Actual Over %  Predicted Over %
              Global NB  4.5000    0.4372 0.1419        71.4605           70.2866
LightGBM Conditional NB  4.5000    0.4327 0.1411        71.4605           70.6023
              Global NB  9.5000    0.5001 0.1632        48.0677           41.8609
LightGBM Conditional NB  9.5000    0.4838 0.1592        48.0677           46.4100
              Global NB 14.5000    0.4184 0.1331        28.8004           24.7801
LightGBM Conditional NB 14.5000    0.4056 0.1294        28.8004           28.2851
              Global NB 19.5000    0.2918 0.0893        15.7045           14.8538
LightGBM Conditional NB 19.5000    0.2853 0.0871        15.7045           16.1072
              Global NB 24.5000    0.1823 0.0529         7.7740            9.0322
LightGBM Conditional NB 24.5000    0.1776 0.0515         7.7740            8.5772
              Global NB 29.5000    0.1077 0.0286         

---
## 5. Model validation and diagnostics
Check the custom likelihood derivatives and inspect the resulting dispersion predictions.

In [ ]:

import numpy as np
import pandas as pd

from scipy.special import (
    digamma,
    polygamma,
    gammaln
)

# ============================================================
# 1. NEGATIVE BINOMIAL NEGATIVE LOG-LIKELIHOOD
# ============================================================

def nb_nll_log_r(z, y, mu):

    r = np.exp(z)

    return (
        -gammaln(y + r)
        + gammaln(r)
        + gammaln(y + 1)
        - r * np.log(r)
        + r * np.log(r + mu)
        - y * np.log(mu)
        + y * np.log(r + mu)
    )


# ============================================================
# 2. ANALYTICAL GRADIENT AND HESSIAN
# ============================================================

def nb_grad_hess_log_r(z, y, mu):

    r = np.exp(z)

    grad_r = (
        -digamma(y + r)
        + digamma(r)
        - np.log(r)
        + np.log(r + mu)
        + (r + y) / (r + mu)
        - 1.0
    )

    hess_r = (
        -polygamma(1, y + r)
        + polygamma(1, r)
        - 1.0 / r
        + 1.0 / (r + mu)
        + (mu - y) / (r + mu)**2
    )

    grad_z = grad_r * r

    hess_z = (
        hess_r * r**2
        + grad_r * r
    )

    return grad_z, hess_z


# ============================================================
# 3. FINITE DIFFERENCE CHECK
# ============================================================

EPS = 1e-4

test_cases = [
    (0, 3.0, -1.0),
    (5, 8.0, 0.0),
    (15, 18.0, 1.0),
    (30, 25.0, 1.5)
]

results = []

for y, mu, z in test_cases:

    analytical_grad, analytical_hess = (
        nb_grad_hess_log_r(z, y, mu)
    )

    numerical_grad = (
        nb_nll_log_r(z + EPS, y, mu)
        - nb_nll_log_r(z - EPS, y, mu)
    ) / (2 * EPS)

    numerical_hess = (
        nb_nll_log_r(z + EPS, y, mu)
        - 2 * nb_nll_log_r(z, y, mu)
        + nb_nll_log_r(z - EPS, y, mu)
    ) / EPS**2

    results.append({
        "Actual Points": y,
        "Expected Points": mu,
        "Log R": z,
        "Analytical Gradient": analytical_grad,
        "Numerical Gradient": numerical_grad,
        "Gradient Error": abs(
            analytical_grad - numerical_grad
        ),
        "Analytical Hessian": analytical_hess,
        "Numerical Hessian": numerical_hess,
        "Hessian Error": abs(
            analytical_hess - numerical_hess
        )
    })


gradient_results_df = pd.DataFrame(results)

print("\n" + "=" * 90)
print("NEGATIVE BINOMIAL GRADIENT / HESSIAN VALIDATION")
print("=" * 90)

print(
    gradient_results_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.8f}"
    )
)

print("\nMAXIMUM ERRORS")
print(
    "Gradient:",
    gradient_results_df["Gradient Error"].max()
)
print(
    "Hessian:",
    gradient_results_df["Hessian Error"].max()
)

assert np.allclose(
    gradient_results_df["Analytical Gradient"],
    gradient_results_df["Numerical Gradient"],
    rtol=1e-4,
    atol=1e-5
), "Gradient check failed"

assert np.allclose(
    gradient_results_df["Analytical Hessian"],
    gradient_results_df["Numerical Hessian"],
    rtol=1e-3,
    atol=1e-4
), "Hessian check failed"

print("\nDerivative checks passed.")



NEGATIVE BINOMIAL GRADIENT / HESSIAN VALIDATION
 Actual Points  Expected Points       Log R  Analytical Gradient  Numerical Gradient  Gradient Error  Analytical Hessian  Numerical Hessian  Hessian Error
             0       3.00000000 -1.00000000           0.48689399          0.48689399      0.00000000          0.19499341         0.19499338     0.00000003
             5       8.00000000  0.00000000          -0.41944209         -0.41944209      0.00000000          0.19231717         0.19231674     0.00000043
            15      18.00000000  1.00000000          -0.42124959         -0.42124959      0.00000000          0.11780730         0.11780728     0.00000002
            30      25.00000000  1.50000000          -0.39522024         -0.39522024      0.00000000          0.11632758         0.11633006     0.00000248

MAXIMUM ERRORS
Gradient: 1.297523755106056e-10
Hessian: 2.4774460699039658e-06

Derivative checks passed.


### 5.2 Prediction-offset diagnostics
Compare reconstructed predictions against stored α values. Agreement confirms implementation consistency, **not** independently that LightGBM's initial score was incorporated correctly during training.

In [ ]:

# ============================================================
# 4. VALIDATE LIGHTGBM PREDICTION OFFSET
# ============================================================

raw_pred = dispersion_model.predict(
    X_dispersion_test,
    raw_score=True
)

# This is the reconstruction used in the previous code.
reconstructed_log_r = (
    INITIAL_LOG_R + raw_pred
)

reconstructed_alpha = np.exp(
    -np.clip(reconstructed_log_r, -8, 12)
)

print("\n" + "=" * 70)
print("LIGHTGBM PREDICTION OFFSET DIAGNOSTICS")
print("=" * 70)

print(f"Initial log(r): {INITIAL_LOG_R:.6f}")
print(f"Global alpha:   {alpha_hat:.6f}")

print("\nRaw model predictions:")
print(
    pd.Series(raw_pred).describe(
        percentiles=[0.05, 0.5, 0.95]
    )
)

print("\nReconstructed alpha:")
print(
    pd.Series(reconstructed_alpha).describe(
        percentiles=[0.05, 0.5, 0.95]
    )
)

print("\nPrevious alpha predictions:")
print(
    pd.Series(alpha_test_conditional).describe(
        percentiles=[0.05, 0.5, 0.95]
    )
)

print(
    "\nMaximum difference from previous alpha:",
    np.max(
        np.abs(
            reconstructed_alpha
            - alpha_test_conditional
        )
    )
)



LIGHTGBM PREDICTION OFFSET DIAGNOSTICS
Initial log(r): 1.103062
Global alpha:   0.331853

Raw model predictions:
count    26833.000000
mean        -0.063273
std          0.853289
min         -1.976944
5%          -1.544296
50%          0.026033
95%          1.276037
max          1.497843
dtype: float64

Reconstructed alpha:
count    26833.000000
mean         0.509828
std          0.463545
min          0.074206
5%           0.092634
50%          0.323326
95%          1.554625
max          2.396193
dtype: float64

Previous alpha predictions:
count    26833.000000
mean         0.509828
std          0.463545
min          0.074206
5%           0.092634
50%          0.323326
95%          1.554625
max          2.396193
dtype: float64

Maximum difference from previous alpha: 4.440892098500626e-16


### 5.3 Dispersion versus expected points
Summarise conditional α, predicted variance, observed squared errors and variance ratios across expected-points ranges. The empirical squared-error comparison also reflects conditional-mean errors.

In [ ]:

# ============================================================
# 5. PREPARE DISPERSION ANALYSIS DATA
# ============================================================

dispersion_analysis_df = pd.DataFrame({

    "actual_points": np.asarray(y_test),

    "predicted_points": np.asarray(mu_test),

    "predicted_alpha": np.asarray(
        alpha_test_conditional
    ),

    "predicted_variance": (
        mu_test
        + alpha_test_conditional * mu_test**2
    ),

    "squared_error": (
        np.asarray(y_test) - mu_test
    ) ** 2
})

dispersion_analysis_df["predicted_sd"] = np.sqrt(
    dispersion_analysis_df["predicted_variance"]
)


# ============================================================
# 6. GROUP BY EXPECTED POINTS
# ============================================================

POINTS_BINS = [
    0, 5, 10, 15, 20, 25, 30, np.inf
]

dispersion_analysis_df["points_bucket"] = pd.cut(
    dispersion_analysis_df["predicted_points"],
    bins=POINTS_BINS,
    include_lowest=True
)

points_dispersion_summary = (
    dispersion_analysis_df
    .groupby("points_bucket", observed=True)
    .agg(
        observations=("actual_points", "size"),
        mean_expected_points=("predicted_points", "mean"),
        mean_actual_points=("actual_points", "mean"),
        mean_alpha=("predicted_alpha", "mean"),
        median_alpha=("predicted_alpha", "median"),
        mean_predicted_sd=("predicted_sd", "mean"),
        mean_nb_variance=("predicted_variance", "mean"),
        empirical_mse=("squared_error", "mean")
    )
    .reset_index()
)

points_dispersion_summary["variance_ratio"] = (
    points_dispersion_summary["empirical_mse"]
    / points_dispersion_summary["mean_nb_variance"]
)

print("\n" + "=" * 100)
print("CONDITIONAL DISPERSION BY EXPECTED POINTS")
print("=" * 100)

print(
    points_dispersion_summary.to_string(
        index=False,
        float_format=lambda x: f"{x:.3f}"
    )
)



CONDITIONAL DISPERSION BY EXPECTED POINTS
points_bucket  observations  mean_expected_points  mean_actual_points  mean_alpha  median_alpha  mean_predicted_sd  mean_nb_variance  empirical_mse  variance_ratio
(-0.001, 5.0]          5090                 3.610               3.402       1.318         1.310              4.415            19.866         15.693           0.790
  (5.0, 10.0]          9374                 7.454               7.476       0.504         0.460              5.745            33.318         28.336           0.850
 (10.0, 15.0]          6440                12.219              12.330       0.237         0.233              6.834            46.899         41.205           0.879
 (15.0, 20.0]          3138                17.274              17.108       0.144         0.142              7.717            59.698         52.106           0.873
 (20.0, 25.0]          1872                22.373              21.880       0.104         0.102              8.587            73.861     

---
## 6. Export V1 model artefacts
Save the fitted mean model, fitted conditional dispersion model and Negative Binomial configuration for later inference.

**Portability note:** A native LightGBM model file can be more reliable than pickling a Booster trained with a custom Python objective. Keep the Python/LightGBM versions and feature schema with the artefacts. Only load trusted pickle files.

**V2 backlog:** Add genuine pregame minutes predictions, retrain the points mean model using OOF minutes estimates, then refit and evaluate dispersion on a fresh chronological holdout.

In [ ]:

import pickle
import json
import os
from datetime import datetime

# ============================================================
# 1. CONFIGURATION
# ============================================================

MODEL_VERSION = "v1"

MODEL_DIR = os.path.join(
    "models",
    "nba_points",
    MODEL_VERSION
)

os.makedirs(MODEL_DIR, exist_ok=True)


# ============================================================
# 2. SAVE LIGHTGBM MEAN MODEL
# ============================================================

mean_model_path = os.path.join(
    MODEL_DIR,
    "points_mean_model.pkl"
)

with open(mean_model_path, "wb") as f:
    pickle.dump(
        final_mean_model,
        f,
        protocol=pickle.HIGHEST_PROTOCOL
    )

print(f"Saved mean model: {mean_model_path}")


# ============================================================
# 3. SAVE LIGHTGBM DISPERSION MODEL
# ============================================================

dispersion_model_path = os.path.join(
    MODEL_DIR,
    "points_dispersion_model.pkl"
)

with open(dispersion_model_path, "wb") as f:
    pickle.dump(
        dispersion_model,
        f,
        protocol=pickle.HIGHEST_PROTOCOL
    )

print(f"Saved dispersion model: {dispersion_model_path}")


# ============================================================
# 4. SAVE NEGATIVE BINOMIAL CONFIGURATION
# ============================================================

nb_config = {

    "version": MODEL_VERSION,

    "created_at": datetime.now().isoformat(),

    "distribution": "negative_binomial_NB2",

    "mean_model": "points_mean_model.pkl",

    "dispersion_model": "points_dispersion_model.pkl",

    "global_alpha": float(alpha_hat),

    "initial_log_r": float(INITIAL_LOG_R),

    "mean_features": list(
        final_mean_model.feature_name_
    ),

    "dispersion_features": list(
        dispersion_model.feature_name()
    ),

    "dispersion_prediction": (
        "log_r = initial_log_r + "
        "dispersion_model.predict(X, raw_score=True)"
    ),

    "dispersion_log_r_clip": [-8.0, 12.0],

    "target": "points"
}

config_path = os.path.join(
    MODEL_DIR,
    "model_config.json"
)

with open(config_path, "w") as f:
    json.dump(
        nb_config,
        f,
        indent=4
    )

print(f"Saved configuration: {config_path}")


# ============================================================
# 5. COMPLETE
# ============================================================

print("\n" + "=" * 70)
print("NBA POINTS PREDICTION V1 SAVED")
print("=" * 70)

print(f"Directory: {MODEL_DIR}")
print("Mean model:       points_mean_model.pkl")
print("Dispersion model: points_dispersion_model.pkl")
print("Configuration:    model_config.json")

print("=" * 70)
